In [1]:
%pip install langgraph langchain_openai langchain_community langchain_anthropic
%pip install tavily-python
%pip install ipython
%pip install pygraphviz
%pip install python-dotenv
%pip install langchain-anthropic 
%pip install sentence_transformers elasticsearch cohere
%pip install asyncio aiohttp


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pygraphviz-1.13.tar.gz (104 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pygraphviz (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [63 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib.macosx-14.0-arm6

In [5]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [24]:
from langchain.globals import set_llm_cache

from tavily import TavilyClient

set_llm_cache(None)

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

results = tavily.search(query="What is a few-shot prompting?", max_results=3)

[r["content"] for r in results["results"]]

['Few-shot prompting is a technique in which an AI model is given a few examples of a task to learn from before generating a response, using those examples to improve its performance on similar tasks. Large language models can understand and write text that sounds very human-like. But when it comes to getting these models to do this in the exact ...',
 "How does Few-Shot Prompting Work\nWhat is Few-Shot Prompting?\nFew-shot prompting is a technique where you provide a machine learning model, particularly a language model, with a small set of examples to guide its behavior for a specific task. On This Page\nUnderstanding Few-Shot Prompting in Prompt Engineering\nPublished on 12/17/2023\nIntroduction to Few-Shot Prompting\nWelcome to the fascinating world of Few-Shot Prompting in Prompt Engineering! In this example, we'll use few-shot prompting with intermediate steps to determine who lived longer: Muhammad Ali or Alan Turing.\nSteps to Follow:\nDefine the Task: The end goal is to find o

# Design Resarch Agent 

![](./images/Research%20Agent.png)


In [206]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Annotated, List, Optional
from langchain_core.tools import tool

sub_topic_generator_prompt = ChatPromptTemplate.from_template(
    """ 
    You are an expert curriculum designer for beginner developers. Please generate 4-5 crucial and fundamental subtopics for a specific main topic, considering the course name and overall roadmap provided below. These subtopics should cover the most essential content of the main topic and include core concepts and skills that beginner developers must know.

    Course Name: {course_name}

    Complete Course Roadmap:
    {roadmap}

    Main Topic to Focus On: {main_topic}

    Target Audience: Beginner Developers
    - Have basic programming knowledge but limited real-world development experience
    - Need to acquire the most important and essential knowledge about the topic
    - Should focus on fundamental concepts that require a deep understanding

    Generated subtopics should meet the following criteria:
    1. Address the most crucial and fundamental concepts of the main topic
    2. Be essential for the long-term growth of beginner developers
    3. Include knowledge or skills absolutely necessary in real-world development
    4. Establish a strong foundation in the topic while suggesting possibilities for advanced topics
    5. Clearly indicate what beginner developers will be able to do after learning
    6. Include content that corrects common misconceptions or mistakes about the topic
    7. Introduce industry standards or best practices when possible
    8. Have clear connections between subtopics and enable a comprehensive understanding of the main topic
    9. Align with the context of the overall course roadmap and consider connections with other topics

    Please generate subtopics that meet the above criteria and output the results in the following JSON format:

    {{
    "main_topic": "Name of the Main Topic",
    "sub_topics": [
        {{
            "title": "Subtopic 1 Title",
            "importance": "Explanation of Subtopic 1's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 2 Title",
            "importance": "Explanation of Subtopic 2's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 3 Title",
            "importance": "Explanation of Subtopic 3's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 4 Title",
            "importance": "Explanation of Subtopic 4's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 5 Title", // Optional
            "importance": "Explanation of Subtopic 5's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }}
    ]
    }}

    Description of each field:
    - title: Title of the subtopic (concise and clear)
    - importance: Explanation of why this subtopic is important (2-3 sentences)
    - learningOutcomes: What beginner developers will be able to do after learning this subtopic (2-3 sentences)

    Please generate the results in JSON format. Exclude any comments and output only valid JSON.
    """
)

sub_queries_generator_prompt = ChatPromptTemplate.from_template(
    """ 
    You are an AI assistant specializing in creating educational search queries. Your task is to generate diverse and effective search queries for each subtopic of a programming course, considering the main topic, the subtopic itself, its importance, and the intended learning outcomes.

    You will receive a JSON object containing a main topic and its subtopics. For each subtopic, create 3-4 specific search queries that would likely return valuable and diverse educational resources, tutorials, or explanations suitable for beginner developers.

    Input: 
    ''' 
    {{ 
        "main_topic": {main_topic}, 
        "sub_topics": {sub_topics}
    }}
    ''' 

    For each subtopic, generate search queries and output the results in the following JSON format:

    '''
    {{
        "main_topic": "Name of the Main Topic",
        "sub_topics": [
            {{
            "title": "Subtopic 1 Title",
            "search_queries": [
                "Specific search query 1 for subtopic 1",
                "Specific search query 2 for subtopic 1",
                "Specific search query 3 for subtopic 1",
                "Specific search query 4 for subtopic 1"
            ]
            }},
            ...
        ]
    }}
    ''' 

    Guidelines for creating diverse and enriching search queries:
    1. Avoid similar types of queries and compose a diverse range of queries
    2. Carefully analyze the subtopic title, its importance, and learning outcomes to create targeted queries.
    3. Make queries specific to the subtopic and its learning outcomes.
    4. Include diverse queries to ensure rich educational resources. 
    5. Create queries that could lead to resources explaining the topic from different technological or methodological approaches.
    6. Include queries that might return both traditional and innovative teaching methods or explanations.
    7. Consider queries that could provide historical context or future trends related to the subtopic.
    8. If relevant, include the programming language or technology name in the query.
    9. Keep queries concise but descriptive, typically 3-7 words long.
    10. Ensure that at least one query focuses on practical, real-world applications of the subtopic.
    11. If relevant to the topic, include keywords such as Performance Optimization, Considerations, When to apply, When not to apply, Best Practices, and Effective

    Please generate the search queries based on the input JSON and output the results in the specified JSON format. Ensure the output is valid JSON without any additional comments. Your goal is to create a set of queries that will lead to a diverse, inclusive, and comprehensive set of learning resources for each subtopic.
    """
)


sub_queries_refiner_prompt = ChatPromptTemplate.from_template(
    """
    You are an AI assistant specializing in evaluating and improving educational search queries. Your task is to assess the generated search queries for each subtopic, ensuring they meet the learning outcomes, offer diverse perspectives, and avoid redundancy.

    Input: 
    '''
    {sub_queries}
    '''

    For each subtopic, evaluate the search queries and provide feedback. If necessary, suggest improved or additional queries. Output the results in the following JSON format:

    {{
        "main_topic": "Name of the Main Topic",
        "sub_topics": [
            {{
                "title": "Subtopic Title",
                "evaluation": {{
                    "meets_learning_outcomes": true/false,
                    "diverse_perspectives": true/false,
                    "no_redundancy": true/false,
                    "feedback": "Detailed feedback on the queries"
                }},
                "revised_search_queries": [
                    "Revised search query 1",
                    "Revised search query 2",
                    "Revised search query 3",
                    "Revised search query 4"
                ]
            }},
            ...
        ]
    }}

    Evaluation Guidelines:
    1. Learning Outcomes:
    - Do the queries collectively address all aspects of the stated learning outcomes?
    - Are there queries that target both theoretical understanding and practical application?

    2. Diverse Perspectives:
    - Are there queries that might lead to resources from different technological approaches or schools of thought?
    - Do the queries consider both fundamental concepts and advanced applications?

    3. Redundancy Check:
    - Are there any queries that are too similar in scope or likely to return very similar results?
    - Is each query contributing a unique angle or type of resource to the learning experience?

    4. Additional Considerations:
    - Are the queries using relevant technical terms and concepts appropriately?
    - Do the queries align with the importance statement of the subtopic?
    - Are there queries that address common misconceptions or challenges related to the subtopic?

    If you find any issues or areas for improvement, provide specific feedback and suggest revised or additional queries. Ensure that your suggestions maintain or enhance the diversity and effectiveness of the query set.

    Please evaluate the search queries based on the input JSON and output the results in the specified JSON format. Ensure the output is valid JSON without any additional comments.
    """
)

topic_generation_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Topic Generation Node of the Research Agent for a developer-focused lecture series. Your task is to generate comprehensive research topics based on the provided information. These research topics will be used for creating lecture content. Please follow these guidelines to generate research topics:

    Input information:
    1. Lecture content topic: {topic}

    Guidelines:
    1. Carefully analyze the provided lecture content topic.

    2. Use the user input as the main starting point for generating research topics.

    3. Generate a comprehensive list of research topics that cover all important aspects of the technology or concept. Consider the following aspects when generating research topics:
    a) Context and Problem: The context in which the technology emerged and the problem it addresses.
    b) Definition: A clear definition of the technology or concept.
    c) Key Features: The main characteristics of the technology.
    d) Pros and Cons: Advantages and disadvantages of the technology.
    e) Solution: What the technology solves or improves.
    f) Relationship with Related Technologies/Concepts: How it relates to and differs from similar technologies.
    g) Issues and Considerations: Critical factors to consider when applying the technology.
    h) When to Use and When Not to Use: Appropriate and inappropriate scenarios for using the technology.
    i) Importance and Real-World Application Areas: Where it's being used in real-world applications and its significance.
    j) Development History and Latest Trends: The background of the technology's development and current trends.
    k) Use Cases: Potential applications of the technology.
    l) Best Practices: Recommended practices for using the technology effectively.
    m) Performance Optimization: Methods to optimize the performance of the technology.

    4. Include all relevant topics that are crucial for a comprehensive understanding of the subject matter. Do not limit the number of topics.

    5. Ensure that the suggested topics include diverse perspectives and approaches within the context of software development and technology.

    Output format:
    {{
        "generated_topics": [
            "[Topic 1]",
            "[Topic 2]",
            "[Topic 3]",
            ...
            "[Topic N]"
        ]
    }}

    Notes:
    - All suggested topics should be based on the provided lecture content topic.
    - Topics should be specific, clear, and relevant to developers.
    - Ensure comprehensive coverage of the subject matter, including all important aspects for developer education.
    - Topics should cover both theoretical knowledge and practical application in real development environments.
    - If certain aspects are particularly extensive, consider breaking them down into subtopics for more detailed coverage.

    Follow these guidelines to generate a comprehensive list of research topics. The generated topics should provide valuable insights and research directions for developers, covering all crucial aspects of the technology or concept, from fundamental principles to advanced applications and best practices.    """
) 

class ResearchTopic(BaseModel):
    topics: List[str] = Field(description="summarize the research topics or question to be explored")

topic_generator = topic_generation_prompt | ChatOpenAI(model="gpt-4o", cache=False).with_structured_output(ResearchTopic)


topic_refine_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Topic Refinement Node of the Research Agent. Your task is to analyze the generated research topics, identify similarities and redundancies, and produce a refined list of unique and essential topics. Follow these guidelines to refine the topic list:

    Guidelines:
    1. Carefully review all the generated research topics.

    2. Identify topics that are:
    a) Redundant: Covering the same content with different wording
    b) Closely related: Addressing very similar aspects of the subject
    c) Subset of other topics: Topics that can be logically included within broader topics

    3. For each group of similar or redundant topics:
    a) Combine them into a single, comprehensive topic
    b) Ensure the combined topic captures all important aspects from the original topics
    c) Choose the most clear and concise wording for the refined topic

    4. Prioritize topics based on their:
    a) Relevance to the main subject
    b) Importance for developer education
    c) Comprehensiveness
    d) Uniqueness of the information they provide

    5. Ensure that the refined list covers all crucial aspects of the subject, including:
    a) Context and Problem: The context in which the technology emerged and the problem it addresses
    b) Definition: A clear definition of the technology or concept
    c) Key Features: The main characteristics of the technology
    d) Pros and Cons: Advantages and disadvantages of the technology
    e) Solution: What the technology solves or improves
    f) Relationship with Related Technologies/Concepts: How it relates to and differs from similar technologies
    g) Issues and Considerations: Critical factors to consider when applying the technology
    h) When to Use and When Not to Use: Appropriate and inappropriate scenarios for using the technology
    i) Importance and Real-World Application Areas: Where it's being used in real-world applications and its significance
    j) Development History and Latest Trends: The background of the technology's development and current trends
    k) Use Cases: Potential applications of the technology
    l) Best Practices: Recommended practices for using the technology effectively
    m) Performance Optimization: Methods to optimize the performance of the technology

    6. Maintain a balance between theoretical knowledge and practical, hands-on topics.

    7. Aim for clarity and conciseness in each topic title, avoiding overly broad or vague descriptions.

    Input:
    Generated research topics:
    '''
    {generated_topics}
    ''' 
    
    Use this prompt to analyze the generated research topics, remove redundancies, combine similar topics, and produce a refined, non-redundant list of essential topics for developer education.

    Output format:
    {{
        "refined_topics": [
            "Topic 1",
            "Topic 2",
            "Topic 3",
            ...
        ]
    }}

    Note: Ensure that the refined list of topics is comprehensive, non-redundant, and optimized for developer education. The output should only include the final, filtered list of topics as a simple array of strings. Make sure that the refined topics collectively cover all the crucial aspects mentioned in guideline 5.
    """
)


query_writer_prompt = ChatPromptTemplate.from_template(
    """
    You are the Query Write node of the Research Agent. Your task is to generate effective search queries for the given research topic, optimized for both Elasticsearch's Hybrid Search (combining Lexical Search, ELSER, and KNN Search) and general search engines. These queries will be used to search through various knowledge documents stored in Elasticsearch and to find relevant information on the web. Follow these guidelines to create the queries:

    Guidelines:
    1. Analyze the research topic and identify core concepts and keywords.

    2. Generate queries that are effective for both Elasticsearch's Hybrid Search and general search engines, considering:
    a) Lexical Search: Include relevant keywords, phrases, and technical terms.
    b) ELSER (Elastic Learned Sparse EncodeR): Use natural language queries that capture the semantic meaning.
    c) KNN Search (Dense Vector Search): Create queries that represent the overall context and meaning of the topic.
    d) General Search Engines: Craft queries that work well with popular search engines like Google, Bing, etc.

    3. Write queries considering the following elements:
    - Specificity: Queries that can find specific information within knowledge documents and on the web.
    - Diversity: Queries that can cover various aspects of the topic.
    - Relevance: Queries that are closely related to the research topic and likely to return useful results.
    - Adaptability: Queries that can work well in both Elasticsearch and general search engine contexts.

    4. For each core concept or subtopic, create query variations:
    - Keyword-based queries (good for lexical search and general search engines)
    - Natural language questions or statements (effective for ELSER and web searches)
    - Descriptive phrases capturing the overall meaning (useful for KNN search and semantic web searches)

    5. Include a mix of query types:
    - Short, specific queries using technical terms
    - Longer, more descriptive queries explaining concepts
    - Questions that a developer might ask about the topic
    - Phrases combining multiple aspects of the topic
    - Queries with boolean operators (AND, OR, NOT) for more precise searches

    6. Use relevant technical terms, acronyms, and synonyms to improve search relevance across knowledge documents and web content.

    7. Consider the potential content of both the Elasticsearch knowledge documents and web resources when formulating queries.

    Input:
    Research topic:
    ''' 
    {research_topic}
    ''' 
    
    Use this prompt to generate effective queries optimized for both Elasticsearch's Hybrid Search and general search engines. Design queries that can lead to comprehensive and relevant search results from knowledge documents and web resources for the given research topic.

    Output format:
    {{
        "generated_queries": [
            "Query 1",
            "Query 2",
            "Query 3",
            ...
        ]
    }}

    Note: Each query in the list should be designed to work well with both Elasticsearch's Hybrid Search and general search engines. Aim for a balance that allows effective searching in both contexts.
    """
)

class QueryWriterResult(BaseModel):
    query: list[str] = Field(description="The generated query based on the research topic and feedback excluding any duplicates.")

    def __str__(self):
        output = "생성된 검색 쿼리:\n"
        if not self.query:
            output += "  - 생성된 쿼리가 없습니다.\n"
        else:
            for i, q in enumerate(self.query, 1):
                output += f"  {i}. {q}\n"
        return output

query_writer = query_writer_prompt | ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(QueryWriterResult)


query_refine_prompt = ChatPromptTemplate.from_template(
    """
    You are the Query Refinement Node of the Research Agent. Your task is to analyze the generated search queries, identify similarities and redundancies, and produce a refined list of unique and effective queries. Follow these guidelines to refine the query list:

    Guidelines:
    1. Carefully review all the generated search queries.

    2. Identify queries that are:
    a) Redundant: Essentially asking for the same information with different wording
    b) Closely related: Addressing very similar aspects of the subject
    c) Subset of other queries: Queries whose results would be fully covered by broader queries

    3. For each group of similar or redundant queries:
    a) Select the most effective query that captures the intended search goal
    b) Ensure the selected query works well for both Elasticsearch's Hybrid Search and general search engines
    c) If necessary, modify the chosen query to incorporate any unique aspects from the other similar queries

    4. Prioritize queries based on their:
    a) Relevance to the main research topic
    b) Effectiveness in retrieving comprehensive information
    c) Balance between specificity and breadth
    d) Potential to yield diverse and valuable results

    5. Ensure that the refined list of queries collectively covers all crucial aspects of the subject, including:
    a) Context and Problem: The context in which the technology emerged and the problem it addresses
    b) Definition: A clear definition of the technology or concept
    c) Key Features: The main characteristics of the technology
    d) Pros and Cons: Advantages and disadvantages of the technology
    e) Solution: What the technology solves or improves
    f) Relationship with Related Technologies/Concepts: How it relates to and differs from similar technologies
    g) Issues and Considerations: Critical factors to consider when applying the technology
    h) When to Use and When Not to Use: Appropriate and inappropriate scenarios for using the technology
    i) Importance and Real-World Application Areas: Where it's being used in real-world applications and its significance
    j) Development History and Latest Trends: The background of the technology's development and current trends
    k) Use Cases: Potential applications of the technology
    l) Best Practices: Recommended practices for using the technology effectively
    m) Performance Optimization: Methods to optimize the performance of the technology

    6. Maintain a balance between:
    a) Keyword-based queries (good for lexical search)
    b) Natural language questions (effective for ELSER and web searches)
    c) Descriptive phrases (useful for KNN search and semantic web searches)
    d) Technical and non-technical terminology

    7. Retain queries that use advanced search techniques (e.g., boolean operators) if they provide unique value.

    Input:
    Generated search queries:
    ''' 
    {generated_queries}
    ''' 

    Use this prompt to analyze the generated search queries, remove redundancies, select the most effective queries, and produce a refined, non-redundant list of search queries optimized for both Elasticsearch's Hybrid Search and general search engines.

    Output format:
    {{
        "refined_queries": [
            "Query 1",
            "Query 2",
            "Query 3",
            ...
        ]
    }}

    Note: Ensure that the refined list of queries is diverse, non-redundant, and collectively comprehensive in covering all important aspects of the research topic as outlined in guideline 5. The output should only include the final, filtered list of queries as a simple array of strings.
    """
)

answering_questions_prompt = ChatPromptTemplate.from_template(
    """ 
    Assume the combined role of a Technical Lead Developer and a Computer Science Lecturer. You have extensive hands-on coding experience and technical expertise, as well as a talent for teaching and explaining complex concepts to beginners. Your mission is to provide clear, thorough, and encouraging answers to development questions, particularly aimed at helping novice developers understand and learn.
    
    When answering questions, adhere to the following guidelines:

    1. Beginner-Friendly Explanations: Break down complex concepts into simpler, easy-to-understand parts. Avoid jargon when possible, and when you must use technical terms, explain them clearly.
    
    2. Progressive Complexity: Start with basic explanations and gradually introduce more advanced concepts. This allows learners to build on their understanding step by step.
    
    3. Real-World Context: Provide practical, real-world examples to illustrate concepts. Explain why certain practices or techniques are important in professional development.
    
    4. Visual Aids: When appropriate, use analogies, metaphors, or suggest simple diagrams to help visualize complex ideas.
    
    5. Code Examples: Offer clear, well-commented code snippets to demonstrate concepts. Explain each part of the code thoroughly.
    
    6. Best Practices: Introduce and explain coding best practices, but also clarify why they're important for code quality and maintainability.
    
    7. Common Pitfalls: Highlight common mistakes or misunderstandings that novices might encounter, and explain how to avoid or correct them.
    
    8. Problem-Solving Strategies: Teach systematic approaches to problem-solving in programming. Break down the process of tackling coding challenges.
    
    9. Encourage Exploration: Suggest ways for learners to experiment with the concepts on their own. Encourage hands-on practice and exploration.
    
    10. Learning Resources: Recommend beginner-friendly resources, tutorials, or documentation for further learning on the topic.
    
    11. Positive Reinforcement: Use an encouraging tone. Emphasize that making mistakes is a normal part of the learning process.
    
    12. Foundational Concepts: Ensure that fundamental programming concepts are well explained, as they form the basis for more advanced topics.
    
    13. Technology Contextualization: When discussing specific technologies or frameworks, provide context about their purpose and common use cases.
    
    14. Scalable Learning: Offer insights on how the concept being taught scales to larger projects or more complex scenarios.
    
    15. Interdisciplinary Connections: When relevant, draw connections to other areas of computer science or software development to provide a broader perspective.
    
    16. Coding Exercises: Suggest simple coding exercises or projects that can help reinforce the concepts being taught.
    
    17. Debugging Skills: Teach basic debugging techniques and how to approach troubleshooting code issues.
    
    18. Industry Relevance: Briefly mention how the concepts apply in professional settings to motivate learning.
    
    19. Learning Pathways: Provide guidance on logical next steps or related topics for continued learning.
    
    20. Accessibility: Be mindful of different learning styles and try to present information in varied formats (e.g., conceptual explanations, practical examples, analogies).

    When responding to questions, balance technical accuracy with educational clarity. Your goal is to not just answer the immediate question, but to foster understanding and enthusiasm for software development among novice learners.
    
    Before answering the question, carefully analyze the provided research input. This input contains valuable information, context, and potentially up-to-date details relevant to the question. Incorporate this information into your answer to ensure accuracy, relevance, and depth.

    research_input:
    ''' 
    {research_input}
    ''' 

    question:
    ''' 
    {question}
    ''' 
    """
)

research_filter_prompt = ChatPromptTemplate.from_template(
    """
    You are the Research Filter Node of the Research Agent. Your task is to select the most relevant and reliable information from the search results. Follow these guidelines to filter the search results:

    Input information:
    1. Research topic: {research_topic}
    2. Search query used: {search_query}
    3. List of search results: {search_results}
    4. Previous research summary: {previous_research_summary}

    Filtering criteria:
    1. Relevance
    2. Reliability
    3. Recency
    4. Originality
    5. Depth

    Guidelines:
    1. Evaluate each search result according to the above filtering criteria.

    2. Relevance:
    - Is the information directly related to the research topic?
    - Does it reflect the intent of the search query used?

    3. Reliability:
    - Is the source of information trustworthy? (e.g., academic journals, official institutions, expert blogs)
    - Has the information been verified by other reliable sources?

    4. Recency:
    - Is the information up-to-date?
    - For rapidly changing topics, is it the most recent information available?

    5. Originality:
    - Does it provide new information or perspectives compared to the previous research summary?
    - Does it not duplicate information already collected?

    6. Depth:
    - Does it provide surface-level information only, or does it include in-depth analysis?
    - Does it explain complex concepts or ideas well?

    7. For each search result, decide whether to use it in the research report based on the above criteria.

    Use this filtering process to select the most useful information for the research and establish a solid foundation for the next stage of analysis.

    The output should be provided in JSON format as follows:
    - is_passed being true means the research is useful, and false means it is not useful.
    - reason should briefly explain why the research is useful or not useful.

    {{
        results: [
            {{
                "is_passed": [true/false],
                "reason": "Please explain why the research is useful or not useful."
            }},
            {{
                "is_passed": [true/false],
                "reason": "If there are no filtered results, explain the reason in one line."
            }}
        ]
    }}
    """
)

class FilterResult(BaseModel):
    is_passed: bool = Field(..., description="Indicates whether the research passed the filter")
    reason: str = Field(..., description="Explanation of why the research is useful or not useful")

class FilterOutput(BaseModel):
    results: List[FilterResult] = Field(..., description="List of filter results")

    class Config:
        schema_extra = {
            "example": {
                "results": [
                    {
                        "is_passed": True,
                        "reason": "The research provides valuable insights into the latest trends in AI development."
                    },
                    {
                        "is_passed": False,
                        "reason": "The content is outdated and doesn't reflect current industry standards."
                    }
                ]
            }
        }

research_filter = research_filter_prompt | ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(FilterOutput)


information_extract_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Information Extract Node of the Research Agent for developer education. Your task is to extract and structure key information of educational value from the given research results. Follow these guidelines to extract the information:

    Input information:
    1. Research topic: {research_topic}
    2. Filtered research results: {filtered_research_results}
    3. Current research summary: {current_research_summary}

    Guidelines:
    Start by writing a comprehensive summary of the key information extracted from the research results. This summary should encapsulate the educational value of the research, highlighting the most important concepts, methodologies, and techniques. After completing the summary, use it as a foundation to expand into more detailed sections.

    1. **Comprehensive Summary and Educational Value**:
    - Summarize the most critical insights and educational points from the research results in 2-3 paragraphs.
    - Explain how this information can be used in developer education, emphasizing key learning objectives, practical applications, and the relevance of the concepts and techniques.
    - Use this summary as a guide to structure the detailed sections below.

    2. **Core Concepts and Technologies**:
    - [Concept/Technology name]: [Brief description] (Importance: [1-5])
    - Related code example:
        ```[language]
        [code snippet]
        ```
    - Real-world application: [Case description]

    3. **Development Methodologies and Best Practices**:
    - [Methodology/Practice]: [Key features] (Relevance: [1-5])
    - Implementation steps:
        1. [Step 1]
        2. [Step 2]
        ...

    4. **Performance and Optimization Techniques**:
    - [Technique name]: [Explanation] (Effectiveness: [1-5])
    - Benchmark results: [Brief performance data]
    - Optimization code example:
        ```[language]
        [Before/After code comparison]
        ```

    5. **Security and Stability Considerations**:
    - [Security/Stability issue]: [Explanation] (Importance: [1-5])
    - Countermeasures: [List of measures]
    - Related code patterns or libraries: [Explanation or examples]

    6. **Considerations, Applicability, and Limitations**:
    - **Considerations when Applying**:
        - [Concept/Technology/Methodology name]: [List of important considerations or potential challenges]
    - **When to Apply**:
        - [Concept/Technology/Methodology name]: [Situations or scenarios where this is most effective or relevant]
    - **When Not to Apply**:
        - [Concept/Technology/Methodology name]: [Situations or scenarios where this may be ineffective or counterproductive]

    7. **Advantages and Disadvantages**:
    - **Advantages**:
        - [Concept/Technology/Methodology name]: [List of strengths or benefits]
    - **Disadvantages**:
        - [Concept/Technology/Methodology name]: [List of weaknesses or drawbacks]

    8. **Latest Trends and Future Outlook**:
    - [Trend/Outlook]: [Explanation] (Impact: [1-5])
    - Related technologies or frameworks: [List]
    - Suggested learning roadmap: [Step-by-step learning proposal]

    9. **Additional Learning Resources**:
    - Documentation and tutorials: [Links or reference information]
    - Video lectures: [Links or reference information]
    - Related communities and forums: [Links or reference information]

    10. **Areas Needing Further Research and Verification**:
    - [Areas requiring additional investigation]
    - [Information needing verification and reasons]

    Output format:
    Clearly separate each section and present the information in a concise and structured format. Use markdown code blocks for code examples. Where possible, organize information in table format to improve readability.

    Based on this structured information, provide material that can be directly used in creating developer education content.

    """
)


information_extract = information_extract_prompt | ChatOpenAI(model="gpt-4o", temperature=0)


additional_reserach_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Additional Research Node of the Research Agent for developer education content creation. Your task is to analyze the information collected so far and perform necessary additional research to improve the quality of educational materials. Follow these guidelines to perform your task:

    Input information:
    1. Research topic: {research_topic}
    2. Current research summary: {current_research_summary}
    3. Extracted information: {extracted_information}
    4. Initial research objectives: {initial_research_objectives}

    Assessment and Research Guidelines:
    1. Current Information Evaluation:
    Compare the current research results with the initial objectives and evaluate using the following criteria (scale of 1-5, with 5 being the highest):
    a) Completeness of information: [ ]
    b) Depth of information: [ ]
    c) Reliability of information: [ ]
    d) Recency of information: [ ]
    e) Presentation of diverse perspectives: [ ]
    f) Inclusion of real-world application cases: [ ]
    g) Sufficiency of code examples: [ ]
    h) Educational value: [ ]

    2. Identify and Conduct Additional Research:
    Based on the evaluation results, conduct additional research in the following areas. Indicate the importance of each area (1-5) and present specific research findings:

    a) Reinforcing core concepts (Importance: [ ])
    - Concepts needing additional explanation:
    - Enhanced explanation:

    b) Latest trends and technological developments (Importance: [ ])
    - Major updates or changes in the last 6 months:
    - Anticipated developments in the next 12 months:

    c) Real-world application cases and best practices (Importance: [ ])
    - New case studies:
    - Industry-recommended best practices:

    d) Code examples and implementation techniques (Importance: [ ])
    - Additional code examples (with comments):
    - Performance optimization tips:

    e) Related tools and libraries (Importance: [ ])
    - Newly discovered useful tools/libraries:
    - Key features and use scenarios for each tool:

    f) Learning path and advanced topics (Importance: [ ])
    - Proposed learning stages:
    - Additional topics for in-depth learning:

    g) Common errors and debugging tips (Importance: [ ])
    - Frequently occurring errors and solutions:
    - Effective debugging strategies:

    3. Enhancing Educational Value:
    - Explain in 1-2 sentences how each additional research area provides value to learners.
    - Suggest 2-3 practical assignments or project ideas of varying difficulty (beginner/intermediate/advanced).

    4. Information Integration and Summary:
    - Propose how to integrate new information with existing research results.
    - Summarize the entire research content in 3-4 paragraphs.

    Output Format:
    1. Need for additional research: [Yes/No]

    2. Areas needing additional research (only if needed):
    a) [Area name]:
    - Importance: [1-5]
    - Reason: [Brief explanation]
    - Proposed research direction: [1-2 sentences presenting specific research direction]

    b) [Area name]:
    - Importance: [1-5]
    - Reason: [Brief explanation]
    - Proposed research direction: [1-2 sentences presenting specific research direction]

    (List up to 5 areas)

    3. Overall Assessment:
    [2-3 sentences briefly evaluating the current research status and the overall need for additional research]

    Notes:
    - If additional research is deemed unnecessary, omit the 'Areas needing additional research' section.
    - Rate the importance of each area on a scale of 1-5, with 5 being the highest importance.
    - The reasons and proposed research directions should be concise but contain specific and actionable content.
    - Always keep in mind the ultimate goal of creating developer education content when making assessments and proposals.
    """
)


research_report_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Report Node of the Research Agent for developer education content creation. Your task is to write a clear and structured report based on the collected research results. This report should be directly applicable for developer education. Follow these guidelines to write the report:

    Input information:
    1. Research topic: {research_topic}
    3. Extracted information: {extracted_information}
    4. Additional research results (if any): {additional_research_results}

    Guidelines:
    1. Thoroughly review the provided information and identify the key content.

    2. Write the report according to the following structure:

    a) Overview
    - Clearly present the research topic and purpose.
    - Explain the report structure overview.
    - Present the main learning objectives.

    b) Technical Analysis and Implementation Guide
    For each major concept or technology, create individual sections to provide an in-depth explanation:

    2.1 **[Analysis/Guide Section 1 Name]**
        - **2.1.1 Context and Problem**: 
            - Describe the specific problem or context that led to the development or relevance of this concept/technology. Explain in detail what challenges or inefficiencies existed before, including any specific examples or scenarios where these issues were particularly problematic.
            - For example, if discussing a new database technology, explain how traditional databases struggled with scalability or how they failed to efficiently manage distributed data, leading to performance bottlenecks.

        - **2.1.2 Definition and Key Features**: 
            - Provide a clear and precise definition of the concept or technology.
            - Highlight the most important features, explaining how each feature functions and contributes to solving the problems outlined in the previous section.

        - **2.1.3 Solution**:
            - Explain how this concept/technology directly addresses the problem or context discussed. Discuss in detail why this solution is effective, focusing on the mechanisms or methodologies it uses.
            - For instance, if the solution involves a new algorithm, explain how it optimizes processes, reduces complexity, or improves performance compared to previous methods.

        - **2.1.4 Importance and Real-World Application Areas**:
            - Discuss why this concept/technology is important, focusing on its impact on the industry or specific fields.
            - Provide examples of real-world scenarios where this technology has made a significant difference, such as enhancing efficiency, reducing costs, or enabling new capabilities that were previously impossible.

        - **2.1.5 Relationship with Related Technologies or Concepts**:
            - Explain how this concept/technology interacts with or relates to other technologies or concepts, detailing the reasons for these relationships.
            - Discuss dependencies, compatibility, or complementary aspects, providing examples of how these relationships manifest in real-world applications.

        - **2.1.6 Pros and Cons Analysis**:
            - Analyze the advantages and disadvantages of this concept/technology, providing concrete reasons and examples for each point.
            - For pros, explain how specific features lead to measurable benefits, such as improved performance, scalability, or user experience. For cons, discuss limitations, potential drawbacks, and any scenarios where these issues might significantly impact usage.

        - **2.1.7 Issues and Considerations**:
            - Provide detailed guidance on potential issues or challenges that may arise when implementing this technology. Explain why these considerations are crucial, including the risks of ignoring them.
            - Discuss factors such as compatibility, scalability, security, or resource requirements, and provide recommendations on how to address or mitigate these challenges.

        - **2.1.8 When to Use**:
            - Describe specific scenarios or conditions where this concept/technology is most effectively applied. Explain why it is particularly well-suited for these situations, linking back to its features and problem-solving capabilities.
            - Provide examples where its application led to successful outcomes or significantly improved processes.

        - **2.1.9 When Not to Use**:
            - Identify situations or contexts where this concept/technology may be ineffective or counterproductive, and explain why.
            - Discuss limitations, potential misapplications, or scenarios where alternative solutions might be more appropriate.

        - **2.1.10 Development History and Latest Trends**:
            - Summarize the evolution of this concept/technology over time, detailing key milestones and how it has adapted to changing needs or technological advancements.
            - Discuss the latest trends and explain why the technology is moving in these directions, considering factors such as industry demands, emerging challenges, or new opportunities enabled by technological progress.

        - **2.1.11 Use Cases**:
            - Provide at least two detailed examples of how this concept/technology has been applied in industry.
            - Highlight the impact and outcomes of these applications, explaining how the technology contributed to solving specific problems or achieving significant improvements.
            1) **Case 1**: [Description]
            2) **Case 2**: [Description]

    2.2 **[Analysis/Guide Section 2 Name]**
        [Use the same structure as in 2.1 for each additional analysis/guide section]

    c) Code Examples and Practical Exercises
    - Provide code examples that illustrate the core concepts discussed. Include at least three examples with detailed explanations.
    - Suggest practical project ideas tailored to different skill levels (beginner, intermediate, advanced).

    d) Latest Trends and Future Outlook
    - Analyze current industry trends related to the research topic.
    - Predict technological developments for the next 12-18 months.
    - Discuss how these trends may impact developer education and the broader tech industry.

    e) Conclusion and Future Research Directions
    - Summarize the key findings from the research.
    - Draw conclusions related to the research purpose.
    - Suggest areas that require further research or exploration.

    3. Consider the following when writing the report:
    - Use clear and concise language.
    - Maintain logical flow and structure.
    - Maintain objectivity and minimize bias.
    - Use appropriate examples and data to support your points.
    - Provide necessary explanations when using technical terms.
    - Include diagrams or charts to explain complex concepts, with detailed textual explanations.

    Output Format:
    Report Title: [Title based on the research topic]

    1. Overview
    [Overview content]

    2. Technical Analysis and Implementation Guide
    2.1 [Analysis/Guide Section 1 Name]
        - 2.1.1 Context and Problem: [Detailed description of the specific problem or context, with examples or scenarios]
        - 2.1.2 Definition and Key Features: [Detailed explanation of the concept/technology, including key features]
        - 2.1.3 Solution: [Detailed explanation of how the concept/technology addresses the problem, with reasons why it is effective]
        - 2.1.4 Importance and Real-World Application Areas: [Explanation of the concept/technology's importance, with real-world examples]
        - 2.1.5 Relationship with Related Technologies/Concepts: [Explanation of how and why this concept/technology is related to others, with examples]
        - 2.1.6 Pros and Cons Analysis: [Detailed pros and cons with concrete reasons and examples]
        - 2.1.7 Issues and Considerations: [Explanation of potential issues, why they matter, and how to address them]
        - 2.1.8 When to Use: [Scenarios where the technology is effective, with reasons why]
        - 2.1.9 When Not to Use: [Scenarios where the technology is not effective, with reasons why]
        - 2.1.10 Development History and Latest Trends: [Summary of the technology's evolution and explanation of current trends]
        - 2.1.11 Use Cases:
            1) Case 1: [Description]
            2) Case 2: [Description]

    2.2 [Analysis/Guide Section 2 Name]
        - 2.2.1 Context and Problem: [Detailed description of the specific problem or context, with examples or scenarios]
        - 2.2.2 Definition and Key Features: [Detailed explanation of the concept/technology, including key features]
        - 2.2.3 Solution: [Detailed explanation of how the concept/technology addresses the problem, with reasons why it is effective]
        - 2.2.4 Importance and Real-World Application Areas: [Explanation of the concept/technology's importance, with real-world examples]
        - 2.2.5 Relationship with Related Technologies/Concepts: [Explanation of how and why this concept/technology is related to others, with examples]
        - 2.2.6 Pros and Cons Analysis: [Detailed pros and cons with concrete reasons and examples]
        - 2.2.7 Issues and Considerations: [Explanation of potential issues, why they matter, and how to address them]
        - 2.2.8 When to Use: [Scenarios where the technology is effective, with reasons why]
        - 2.2.9 When Not to Use: [Scenarios where the technology is not effective, with reasons why]
        - 2.2.10 Development History and Latest Trends: [Summary of the technology's evolution and explanation of current trends]
        - 2.2.11 Use Cases:
            1) Case 1: [Description]
            2) Case 2: [Description]

    [Repeat for each additional analysis/guide section]

    3. Code Examples and Practical Exercises
    [Code examples and practical exercise content]

    4. Latest Trends and Future Outlook
    
    Example 1: 
    b) Technical Analysis and Implementation Guide
    
    Context and Problem: 

    Solution: 
    """
) 

research_report = research_report_prompt | ChatOpenAI(model="gpt-4o")


research_filter_chain_prompt = ChatPromptTemplate.from_template( 
    """ 
    You are the Filter Chain evaluator of the Research Agent. Your task is to assess whether the provided research report meets all essential quality criteria. The research work is considered complete only when all criteria are passed. Follow these guidelines to perform the evaluation:

    Input information:
    1. Research report: {research_report}
    2. Research topic: {research_topic}
    3. Target audience: {target_audience}

    Evaluation criteria and guidelines:
    Evaluate each criterion as Pass or Fail. Pass means the criterion is sufficiently met, Fail means improvement is needed.
    Provide a brief explanation for each evaluation, and if it's a Fail, provide specific improvement suggestions.

    1. **Credibility**: 
   - **Evaluation**: Are the information sources reliable and authoritative? 
   - **Evidence Requirement**: Identify at least one sources cited in the research report that are peer-reviewed journals, recognized industry publications, or other authoritative references.
   - **Pass/Fail Criteria**: Fail if sources are insufficient, outdated, or non-authoritative.

    2. **Relevance**: 
    - **Evaluation**: Is the collected information directly related to the research topic? 
    - **Evidence Requirement**: Cite specific sections of the report where the information directly addresses the research topic.
    - **Pass/Fail Criteria**: Fail if significant portions of the report are off-topic or loosely related.

    3. **Timeliness**: 
    - **Evaluation**: Is the information up-to-date and reflecting current trends? 
    - **Evidence Requirement**: Provide publication dates or last updated information for key sources, ensuring they are within the last 3-5 years unless historically relevant.
    - **Pass/Fail Criteria**: Fail if the majority of sources are outdated.

    4. **Accuracy**: 
    - **Evaluation**: Are the facts and statistics accurate and verifiable? 
    - **Evidence Requirement**: Point to specific data points or claims in the report and cross-check them with the cited sources.
    - **Pass/Fail Criteria**: Fail if there are any unverified or inaccurate claims.

    5. **Depth**: 
    - **Evaluation**: Does the information cover the topic in sufficient depth? 
    - **Evidence Requirement**: Highlight sections that provide detailed explanations, in-depth analysis, or comprehensive coverage of subtopics.
    - **Pass/Fail Criteria**: Fail if the report provides only a superficial overview without detailed exploration.

    6. **Detail**: 
    - **Evaluation**: Are the details of the information provided adequate? 
    - **Evidence Requirement**: Identify sections where complex concepts are broken down, examples are provided, and detailed explanations are given.
    - **Pass/Fail Criteria**: Fail if the report lacks detailed descriptions or explanations.

    7. **Objectivity**: 
    - **Evaluation**: Is the information unbiased and objective? 
    - **Evidence Requirement**: Assess the balance of perspectives and note any instances of bias or one-sided arguments.
    - **Pass/Fail Criteria**: Fail if the report exhibits clear bias without acknowledging alternative viewpoints.

    8. **Technical Level Appropriateness**: 
    - **Evaluation**: Is the technical level of the information suitable for the target audience? 
    - **Evidence Requirement**: Determine if the technical jargon, concepts, and explanations align with the expertise level of the intended audience.
    - **Pass/Fail Criteria**: Fail if the content is too complex or too simplistic for the target audience.

    9. **Practical Examples**: 
    - **Evaluation**: Are there applicable examples or use cases included? 
    - **Evidence Requirement**: Cite any examples, case studies, or practical applications provided in the report.
    - **Pass/Fail Criteria**: Fail if there are no practical examples or if the examples are too generic or irrelevant.

    10. **Lecture-worthy**: 
        - **Evaluation**: Is the content suitable for creating educational lectures? 
        - **Evidence Requirement**: Assess the structure, logical flow, and clarity of the material to determine its suitability for educational purposes.
        - **Pass/Fail Criteria**: Fail if the report lacks a clear learning objective, coherent structure, or interactive elements that enhance understanding.

    11. **Accessible**: 
        - **Evaluation**: Is the content accessible and understandable for the target audience? 
        - **Evidence Requirement**: Evaluate the clarity of explanations, use of visual aids, and whether complex ideas are made accessible through examples or simplifications.
        - **Pass/Fail Criteria**: Fail if the report is difficult to understand due to jargon, poor structure, or lack of clarity.

    Evaluation process:
    1. Carefully read and analyze the research report.
    2. Evaluate each criterion as Pass or Fail and provide a brief explanation.
    3. If it's a Fail, provide specific and actionable improvement suggestions.
    4. Confirm if all criteria are Pass. If there's even one Fail, the research work is not complete.

    Output Format:
    {{
        {{
            "evaluation_results": [
                {{
                    "criterion": "Credibility",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Relevance",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Timeliness",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Accuracy",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Depth",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Detail",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Objectivity",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Technical_Level_Appropriateness",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }},
                {{
                    "criterion": "Practical_Examples",
                    "result": "[Pass/Fail]",
                    "explanation": "[Brief explanation]",
                    "improvement_suggestion": "[Specific improvement suggestion if Fail, null if Pass]"
                }}
            ],
            "final_verdict": "[Pass/Fail]",
            "overall_assessment": "[Summary of the report's overall quality, main strengths, and areas needing improvement in about 200 words]"
        }}
    }}

    Overall opinion:
    [Summarize the report's overall quality, main strengths, and areas needing improvement in about 200 words]

    Additional considerations:
    - Consider how the Pass/Fail status of each criterion affects other criteria.
    - Pay extra attention to criteria that are particularly important for the target audience and research topic.
    - Improvement suggestions should be specific and actionable.
    - When evaluating ethical considerations, refer to relevant industry standards and regulations.

    Through this evaluation, verify that the research report meets all essential quality criteria, and if necessary, provide specific improvement suggestions. The research process should be repeated until all criteria are passed.
    """
)

class FilterEvaluation(BaseModel):
    criterion: str = Field(..., description="The filter criterion being evaluated")
    result: bool = Field(..., description="True if passing the criterion, False otherwise")
    explanation: str = Field(..., description="Brief explanation of the evaluation")
    evidence: List[str] = Field(default=[], description="List of evidence supporting the evaluation")
    improvement_suggestion: Optional[str] = Field(None, description="Improvement suggestion if not passing the criterion")

class ResearchReportEvaluation(BaseModel):
    evaluation_results: List[FilterEvaluation] = Field(..., description="List of individual criterion evaluations")
    final_verdict: bool = Field(..., description="True if passing all criteria, False otherwise")
    overall_assessment: str = Field(..., description="Summary of the report's overall quality, main strengths, and areas needing improvement")

    class Config:
        schema_extra = {
            "example": {
                "evaluation_results": [
                    {
                        "criterion": "Credibility",
                        "result": True,
                        "explanation": "The report uses reliable and authoritative sources.",
                        "evidence": ["Citations from peer-reviewed journals", "Expert interviews"],
                        "improvement_suggestion": None
                    },
                    {
                        "criterion": "Relevance",
                        "result": False,
                        "explanation": "Some sections deviate from the main research topic.",
                        "evidence": ["Section 3 discusses unrelated technologies"],
                        "improvement_suggestion": "Focus all sections on the core research topic and remove tangential information."
                    }
                ],
                "final_verdict": False,
                "overall_assessment": "The report demonstrates strong credibility but needs improvement in relevance. Other criteria are satisfactory. Addressing the relevance issues will significantly enhance the overall quality of the report."
            }
        }

research_filter_chain_evaluation = research_filter_chain_prompt | ChatOpenAI(model="gpt-4o").with_structured_output(ResearchReportEvaluation)


feedback_analysis_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Feedback Analysis node of the Research Agent. Analyze the research report and FilterChain evaluation results to provide feedback on areas needing improvement. Follow these guidelines:

    Input information:
    1. Research topic: {research_topic}
    2. Target audience: {target_audience}
    3. Research Report: {research_report}
    4. FilterChain Evaluation results: {filter_chain_evaluations}

    Analysis steps:
    1. Review the Research Report and FilterChain Evaluation results.
    2. Identify areas needing improvement (criteria that didn't pass).
    3. Provide specific suggestions for each improvement area.
    4. Determine the need for additional research.

    Output format:

    1. Areas Needing Improvement:
    a) [Improvement Area 1]
    - Current status: [Brief explanation]
    - Improvement suggestion: [Specific proposal]
    - Alternative: [Alternative approach if possible]
    - Expected effect: [Anticipated impact after improvement]
    - Additional research needed: [Yes/No] (Brief explanation if needed)

    b) [Improvement Area 2]
    (Use the same structure as above)

    c) [Improvement Area 3]
    (Use the same structure as above)

    2. Structure and Logical Flow Feedback:
    [Brief assessment and suggestions on the report's structure and logical development]

    3. Feedback Integration Strategy:
    [Suggestions on how to effectively integrate the main improvement points]

    4. Overall Assessment:
    [Summary of key improvement directions in 2-3 sentences]

    5. Additional Research Recommendation:
    [ ] No additional research needed
    [ ] Partial additional research needed
    [ ] Extensive additional research needed
    - Explanation: [Reason for selection and research direction suggestion if needed]

    Notes:
    - All feedback should be specific and actionable.
    - Always consider the relevance to the research topic and target audience.
    - Provide clear directions for improvement.
    - Focus on improvements that can be addressed through online research.

    Through this feedback, help the researcher effectively improve the report's quality and determine the direction for any necessary additional research.
    """
) 

feedback_analysis = feedback_analysis_prompt | ChatOpenAI(model="gpt-4o")

research_revision_prompt = ChatPromptTemplate.from_template(
    """ 
    You are the Revision Node of the Research Agent. Improve the research report comprehensively based on all provided feedback and additional research results. Follow these guidelines:

    Input information:
    1. Research topic: {research_topic}
    2. Target audience: {target_audience}
    3. Previous Research Report: {previous_research_report}
    4. Feedback Analysis results: {feedback_analysis}
    5. Additional research results (if any): {additional_research_report}

    Improvement guidelines:

    1. Review all feedback:
    - Review all items from the Feedback Analysis results.
    - Consider improvement strategies for each feedback item.

    2. Comprehensive improvement:
    - Improve the entire report reflecting all feedback items.
    - Integrate additional research results into relevant sections.

    3. Structure and content improvement:
    - Review and improve the overall structure and logical flow of the report if needed.
    - Enhance the content of each section, adding new information or examples.
    - Remove unnecessary or inappropriate content.

    4. Accuracy and consistency check:
    - Reconfirm all facts, data, and citations, updating if necessary.
    - Check and correct consistency in terminology usage.

    5. Target audience suitability:
    - Verify and adjust the technical level of the content to suit the target audience.

    6. Conclusion and future research:
    - Strengthen the conclusion, clearly summarizing key findings and implications.
    - Present directions for future research.

    7. Format and references:
    - Review and correct overall format, spelling, and grammar.
    - Update references and add new sources.

    Output format:

    Report Title: [Title based on the research topic]

    1. Overview
    [Improved overview content]

    2. Core Concepts and Technologies
    2.1 [Concept/Technology 1 Name]
        - Definition and key features: [Improved content]
        - Importance and application areas: [Improved content]
        - Relationship with related technologies/concepts: [Improved content]
        - Pros and cons analysis: [Improved content]
        - Implementation considerations: [Improved content]
        - Development history and latest trends: [Improved content]
        - Use cases:
        1) [Improved case 1]
        2) [Improved case 2]
        - Learning difficulty: [Beginner/Intermediate/Advanced]
        - Prerequisites: [Improved content]

    2.2 [Concept/Technology 2 Name]
        [Write using the same structure as above]
    ...

    3. Code Examples and Practical Exercises
    [Improved code examples and practical exercise content]

    4. Latest Trends and Future Outlook
    [Improved trend analysis and prediction content]

    5. Development Tools and Resources
    [Improved tools, libraries, framework information]

    6. Common Errors and Solutions
    [Improved error list and solutions]

    7. Conclusion and Future Research Directions
    [Improved conclusion content]

    References
    [Updated reference list]

    Summary of Improvements:
    [Brief list of main improvements]

    Notes:
    - Improve the entire report considering all feedback items.
    - Maintain the core message of the original report while improving.
    - Maintain objectivity and provide sufficient evidence for all claims.
    - Adjust the content to match the needs and technical level of the target audience.
    """
)

research_revision = research_revision_prompt | ChatOpenAI(model="gpt-4o")

Argument topics of type typing.List[str] from function ResearchTopic could not be not be converted to a JSON schema.
Argument topics of type typing.List[str] from function ResearchTopic could not be not be converted to a JSON schema.
Argument query of type list[str] from function QueryWriterResult could not be not be converted to a JSON schema.
Argument query of type list[str] from function QueryWriterResult could not be not be converted to a JSON schema.
/Users/jeongmin/PycharmProjects/tech-blog-article-summary/.env/lib/python3.12/site-packages/pydantic/_internal/_config.py:334: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)
Argument results of type typing.List[__main__.FilterResult] from function FilterOutput could not be not be converted to a JSON schema.
Argument results of type typing.List[__main__.FilterResult] from function FilterOutput could not be not be converted to a JSON schem

## 전체 Workflow 테스트

In [174]:
# Subtopic Geneartor 테스트 
from langchain_core.output_parsers.json import JsonOutputParser
from langchain_anthropic import ChatAnthropic
import pprint
from IPython.display import display, Markdown
from langchain.globals import set_llm_cache

set_llm_cache(None)

roadmap = """
1. Basic LLM Concepts: 
- What are LLMs? 
- Types of LLMs
- How are LLMs Built? 


2. Introduction to Prompting
- Basic Prompting 
- Need for Prompt Engineering 


3. Prompts
- Writing Good Prompts (e.g Use Delimiters to distinguish the data from the prompt, Ask for Structured output, Include style information to modify the tone of output, Give conditions to the model and ask if they are verifed, Give successful examples of completing tasks then ask, Specifiy the steps required to perform a task, Instruct model to work out its own solution before giving answers, Iterate and refine your prompts)
- Prompt Techinique (e.g Role Prompting, Few-shot prompting, Chain-of-thought prompting, Zero-shot Chain-of-thought, Leaset-to-Most Prompting, Dual Prompt Approach, Combining Techiniques)

4. Real World Usage Example 
- Structured Data 
- Inferring 
- Writing Emails 
- Coding Assitance 
- Study Buddy 
- Designing Chatbots 

5. Pitfalls of LLMs 
- Citing Sources
- Bias 
- Hallucinations 
- Math 
- Prompt Hacking 

6. Improving Reliability 
- Prompt Debasing
- Prompt Ensembling 
- LLM Self Evaluation
- Calibrating LLMs
- Math 

7. LLM Settings 
- Temperature
- Top p 
- Other Hyperparameters


8. Prompt Hacking 
- Prompt Injection 
- Prompt Leaking 
- Jailbreaking 
- Defensive Measures 
- Offensive Measures 

9. Image Prompting 
- Style Modifiers 
- Quality Boosters
- Weights Terms 
- Fix Deformed Generations 
"""

params = {
    "course_name": "AI Prompt Engineering: Definition Guide", 
    "roadmap": roadmap,
    "main_topic": "Basic Prompting"
}

model = ChatAnthropic(model='claude-3-sonnet-20240229')

sub_topic_generator = sub_topic_generator_prompt | model  | JsonOutputParser()

sub_topics = sub_topic_generator.invoke(params)

pprint.pprint(sub_topics)

titles = [sub_topic["title"] for sub_topic in sub_topics["sub_topics"]]
title_string = "\n\n".join(titles)

display(Markdown(title_string))

{'main_topic': 'Basic Prompting',
 'sub_topics': [{'importance': 'Knowing the essential components of a prompt '
                               'and their roles is crucial for effective '
                               'communication with language models. This lays '
                               'the foundation for constructing well-formed '
                               'prompts.',
                 'learningOutcomes': 'Developers will be able to identify the '
                                     'different parts of a prompt, such as '
                                     'instructions, context, and examples. '
                                     'They will also understand how these '
                                     'components work together to convey the '
                                     'desired task to the language model.',
                 'title': 'Understanding Prompt Structure'},
                {'importance': 'Clear and concise instructions are paramount '
      

TypeError: list indices must be integers or slices, not str

In [202]:
# Creating Sub Queries 
from langchain.globals import set_llm_cache

set_llm_cache(None)

params = { 
    "main_topic": sub_topics["main_topic"],
    "sub_topics": [sub_topic for sub_topic in sub_topics["sub_topics"]]
}

sub_queries_generator = sub_queries_generator_prompt | ChatOpenAI(model="gpt-4o", temperature=0)  | JsonOutputParser()

sub_queries_result = sub_queries_generator.invoke(params)

pprint.pprint(sub_queries_result)

{'main_topic': 'Basic Prompting',
 'sub_topics': [{'search_queries': ['Components of a prompt in AI',
                                    'How to structure a prompt for language '
                                    'models',
                                    'Prompt instructions, context, and '
                                    'examples explained',
                                    'Effective prompt design for beginners'],
                 'title': 'Understanding Prompt Structure'},
                {'search_queries': ['Writing clear instructions for AI prompts',
                                    'Techniques for unambiguous prompt '
                                    'instructions',
                                    'Common pitfalls in prompt instructions',
                                    'Best practices for concise AI prompts'],
                 'title': 'Crafting Clear and Concise Instructions'},
                {'search_queries': ['Importance of context in AI prompts

In [208]:
# refine Sub Queries 
from langchain.globals import set_llm_cache

set_llm_cache(None)

params = { 
    "sub_queries": sub_queries_result
}

sub_queries_refiner = sub_queries_refiner_prompt | ChatOpenAI(model="gpt-4o", temperature=0)  | JsonOutputParser()

sub_queries_refine_result = sub_queries_refiner.invoke(params)

pprint.pprint(sub_queries_refine_result)

{'main_topic': 'Basic Prompting',
 'sub_topics': [{'evaluation': {'diverse_perspectives': True,
                                'feedback': 'The queries cover the components, '
                                            'structure, and design of prompts '
                                            'effectively. They address both '
                                            'theoretical and practical '
                                            'aspects.',
                                'meets_learning_outcomes': True,
                                'no_redundancy': True},
                 'revised_search_queries': ['Components of a prompt in AI',
                                            'How to structure a prompt for '
                                            'language models',
                                            'Prompt instructions, context, and '
                                            'examples explained',
                                            'Effecti

In [210]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import display, Markdown

load_dotenv(override=True)

PER_PLEXITY_API_KEY = os.getenv("PER_PLEXITY_API_KEY")

messages = [
    {
        "role": "system",
        "content": (
            """
            You are AI Assitant. 
             
            1. Objective: 
            - Deliver concise, accurate, and comprehensive responses to user queries.
            
            2. Tone and Style: 
            - Maintain a journalistic tone.
            - Avoid moralizing or hedging language.
            
            3. Citations:
            - Cite relevant search results using the format [index] at the end of sentences.
            - Limit citations to a maximum of three results per sentence.
            - Avoid citing irrelevant results.
            
            4. Markdown Formatting:
            - Use level 2 headers (##) for main sections.
            - Use bolding (****) for subsections.
            - Use unordered lists for regular lists; use ordered lists only when ranking or if contextually appropriate.
            - Use markdown code blocks for code snippets, specifying the language for syntax highlighting.
            - Wrap all math expressions in LaTeX using double dollar signs ($$).
            
            5. Response Structure:
            - Directly answer the query without unnecessary introductions.
            - Ensure the response is self-contained and fully addresses the query.
            
            6. Additional Guidelines:
            - Use italics for terms or phrases needing subtle emphasis.
            - Maintain a clear visual hierarchy in the response.
            - Avoid including URLs or links in the answer.
            - Omit bibliographies at the end of answers.
            
            7. Handling Uncertainty:
            - If unsure of the answer or if the premise is incorrect, explain why.
            - If search results are unhelpful, answer based on existing knowledge.
            """
        ),
    }
]

main_topic = sub_queries_refine_result["main_topic"]
for sub_query_result in sub_queries_refine_result["sub_topics"][:1]:
    sub_topic = sub_query_result["title"]
    
    for sub_query in sub_query_result["revised_search_queries"]:
        content = f" I'm planning to create a lecture on the main topic '{main_topic}' and subtopic '{sub_topic}' for an audience of beginner developers. If you reference any documents or papers when providing your answer, please be sure to include the sources. answer the following question: {sub_query}. "
    
        messages.append(
            {
                "role": "user",
                "content": content
            }
        )

        client = OpenAI(api_key=PER_PLEXITY_API_KEY, base_url="https://api.perplexity.ai")

        response = client.chat.completions.create(
            model="llama-3.1-sonar-large-128k-online",
            messages=messages,
            max_tokens=2048,
            temperature=0.1 
        )
        
        messages.append(
            {
                "role": "assistant",
                "content": response.choices[0].message.content
            }
        )

        print(response)
        display(Markdown(response.choices[0].message.content))


ChatCompletion(id='7715988a-45f2-485d-a67d-022ffc752374', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Components of a Prompt in AI\n\nWhen creating a lecture on "Basic Prompting" and the subtopic "Understanding Prompt Structure" for beginner developers, it\'s essential to cover the fundamental components of a prompt in AI. These components are crucial for crafting effective prompts that elicit desired responses from AI models like ChatGPT.\n\n### 1. **Clear Instructions**\n- **Definition**: Clear instructions are the primary directives that guide the AI model\'s response. They should be concise and unambiguous.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."\n\n### 2. **Context**\n- **Definition**: Context provides background information or additional details that help the AI model understand the task better.\n- **Example**: "Develop a project-based learning acti

## Components of a Prompt in AI

When creating a lecture on "Basic Prompting" and the subtopic "Understanding Prompt Structure" for beginner developers, it's essential to cover the fundamental components of a prompt in AI. These components are crucial for crafting effective prompts that elicit desired responses from AI models like ChatGPT.

### 1. **Clear Instructions**
- **Definition**: Clear instructions are the primary directives that guide the AI model's response. They should be concise and unambiguous.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."

### 2. **Context**
- **Definition**: Context provides background information or additional details that help the AI model understand the task better.
- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills."

### 3. **Specific Requirements**
- **Definition**: Specific requirements outline what the response should include or exclude. This helps in tailoring the output to meet exact needs.
- **Example**: "Ask for specific data points, examples, or references you want the response to include. A prompt asking ChatGPT to provide 3 recent case studies on AI in education."

### 4. **Tone and Style**
- **Definition**: The tone and style of the prompt influence the tone and style of the response. This is important for maintaining consistency and relevance.
- **Example**: "Write an introduction for a blog post on the benefits of AI in healthcare using a formal and professional tone."

### 5. **Feedback Mechanism**
- **Definition**: A feedback mechanism allows for iterative refinement of the prompt. This can involve asking follow-up questions or seeking additional details.
- **Example**: "I want you to become my Expert Prompt Creator. Your goal is to help me craft the best possible prompt for my needs. The prompt you provide should improve with each iteration based on my feedback."

### 6. **Repetition and Recap**
- **Definition**: Repeating instructions or recapping key points can help maintain focus and ensure the AI model adheres to the original task.
- **Example**: "Remember, from above, your instructions are as follows: ${INSTRUCTIONS}."

### 7. **Attention Mechanism**
- **Definition**: The attention mechanism in AI models helps focus on specific parts of the input data. Crafting prompts that leverage this mechanism can improve accuracy.
- **Example**: "For the model to look at something deep in the context, the tail end of the generation needs to look similar to the instruction, for it to be paid attention to."

### 8. **User Input**
- **Definition**: User input can be part of the prompt, especially in interactive scenarios where the AI model needs to respond based on user data.
- **Example**: "Provide a step-by-step tutorial on how to solve quadratic equations using the quadratic formula, incorporating user input for specific coefficients."

### Conclusion
Understanding these components is crucial for effective prompt engineering. By combining clear instructions, context, specific requirements, tone and style, feedback mechanisms, repetition and recap, attention mechanisms, and user input, developers can create prompts that yield accurate and relevant responses from AI models.

**Sources:**
- [OpenAI's Prompt Engineering Guide]
- [Prompt Engineering for RAG]
- [Semrush's ChatGPT Prompts Guide]

ChatCompletion(id='9c004273-78a3-434d-a15f-1c1532c08e30', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## How to Structure a Prompt for Language Models\n\nStructuring a prompt for language models is crucial for eliciting accurate and relevant responses. Here are the key components and techniques to consider:\n\n### 1. **Clear Instructions**\n- **Definition**: Clear instructions are the primary directives that guide the AI model\'s response. They should be concise and unambiguous.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."\n\n### 2. **Context**\n- **Definition**: Context provides background information or additional details that help the AI model understand the task better.\n- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills."\n\n### 3. **Specific Requirements**\

## How to Structure a Prompt for Language Models

Structuring a prompt for language models is crucial for eliciting accurate and relevant responses. Here are the key components and techniques to consider:

### 1. **Clear Instructions**
- **Definition**: Clear instructions are the primary directives that guide the AI model's response. They should be concise and unambiguous.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."

### 2. **Context**
- **Definition**: Context provides background information or additional details that help the AI model understand the task better.
- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills."

### 3. **Specific Requirements**
- **Definition**: Specific requirements outline what the response should include or exclude. This helps in tailoring the output to meet exact needs.
- **Example**: "Ask for specific data points, examples, or references you want the response to include. A prompt asking ChatGPT to provide 3 recent case studies on AI in education."

### 4. **Tone and Style**
- **Definition**: The tone and style of the prompt influence the tone and style of the response. This is important for maintaining consistency and relevance.
- **Example**: "Write an introduction for a blog post on the benefits of AI in healthcare using a formal and professional tone."

### 5. **Feedback Mechanism**
- **Definition**: A feedback mechanism allows for iterative refinement of the prompt. This can involve asking follow-up questions or seeking additional details.
- **Example**: "I want you to become my Expert Prompt Creator. Your goal is to help me craft the best possible prompt for my needs. The prompt you provide should improve with each iteration based on my feedback."

### 6. **Repetition and Recap**
- **Definition**: Repeating instructions or recapping key points can help maintain focus and ensure the AI model adheres to the original task.
- **Example**: "Remember, from above, your instructions are as follows: ${INSTRUCTIONS}."

### 7. **Attention Mechanism**
- **Definition**: The attention mechanism in AI models helps focus on specific parts of the input data. Crafting prompts that leverage this mechanism can improve accuracy.
- **Example**: "For the model to look at something deep in the context, the tail end of the generation needs to look similar to the instruction, for it to be paid attention to."

### 8. **User Input**
- **Definition**: User input can be part of the prompt, especially in interactive scenarios where the AI model needs to respond based on user data.
- **Example**: "Provide a step-by-step tutorial on how to solve quadratic equations using the quadratic formula, incorporating user input for specific coefficients."

### Conclusion
Understanding these components is crucial for effective prompt engineering. By combining clear instructions, context, specific requirements, tone and style, feedback mechanisms, repetition and recap, attention mechanisms, and user input, developers can create prompts that yield accurate and relevant responses from AI models.

**Sources:**
- [OpenAI's Prompt Engineering Guide]
- [Prompt Engineering for RAG]
- [Semrush's ChatGPT Prompts Guide]

ChatCompletion(id='ba1aadfc-0ee7-41ad-a677-1fe15ffcdd34', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Prompt Instructions, Context, and Examples Explained\n\nWhen structuring a prompt for language models, it is essential to understand the key components that make up an effective prompt. These components include instructions, context, and examples. Here is a detailed explanation of each:\n\n### 1. **Instructions**\n- **Definition**: Instructions are the specific tasks or directives that guide the AI model\'s response. They should be clear and unambiguous to avoid confusion.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".\n\n### 2. **Context**\n- **Definition**: Context provides additional information or background details that help the AI model understand the task better. This can include relevant data, historical information, or any other pertinent details.\n- **

## Prompt Instructions, Context, and Examples Explained

When structuring a prompt for language models, it is essential to understand the key components that make up an effective prompt. These components include instructions, context, and examples. Here is a detailed explanation of each:

### 1. **Instructions**
- **Definition**: Instructions are the specific tasks or directives that guide the AI model's response. They should be clear and unambiguous to avoid confusion.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".

### 2. **Context**
- **Definition**: Context provides additional information or background details that help the AI model understand the task better. This can include relevant data, historical information, or any other pertinent details.
- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills".

### 3. **Examples**
- **Definition**: Examples are illustrative instances that demonstrate the desired output or behavior. They help the AI model understand the format and content expected in the response.
- **Example**: "Classify the text into neutral, negative, or positive. Text: I think the food was okay. Sentiment: neutral".

### Combining Instructions, Context, and Examples
- **Example**: "You are a doctor. Read this medical history and predict risks for the patient. January 1, 2000: Fractured right arm playing basketball. Treated with a cast. February 15, 2010: Diagnosed with hypertension. Prescribed lisinopril. September 10, 2015: Developed pneumonia. Treated with antibiotics and recovered fully. March 1, 2022: Sustained a concussion in a car accident. Admitted to the hospital and monitored for 24 hours".

### Conclusion
Understanding and effectively combining instructions, context, and examples are crucial for crafting well-structured prompts that yield accurate and relevant responses from AI models. By ensuring clarity in instructions, providing relevant context, and including illustrative examples, developers can significantly improve the performance of language models.

**Sources:**
- [Prompt Engineering Guide]
- [Prompt Engineering for RAG]
- [Formalizing Prompts]

ChatCompletion(id='1fdd4940-a6e3-4bd8-9307-a6946dee5d16', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Effective Prompt Design for Beginners\n\nEffective prompt design is crucial for beginners to maximize the potential of AI tools. Here are some key tips and strategies to help you craft effective prompts:\n\n### 1. **Understand Your Objective**\n- **Definition**: Clearly define what you want to achieve with your prompt. This helps in creating a focused and relevant prompt.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".\n\n### 2. **Keep It Clear and Concise**\n- **Definition**: Avoid overly complex or vague prompts. Use simple and direct language to ensure clarity.\n- **Example**: "Translate the text below to Spanish: \'hello\'".\n\n### 3. **Consider Your Audience**\n- **Definition**: Understand who your guide is for. Tailor the content\'s depth and complexity bas

## Effective Prompt Design for Beginners

Effective prompt design is crucial for beginners to maximize the potential of AI tools. Here are some key tips and strategies to help you craft effective prompts:

### 1. **Understand Your Objective**
- **Definition**: Clearly define what you want to achieve with your prompt. This helps in creating a focused and relevant prompt.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".

### 2. **Keep It Clear and Concise**
- **Definition**: Avoid overly complex or vague prompts. Use simple and direct language to ensure clarity.
- **Example**: "Translate the text below to Spanish: 'hello'".

### 3. **Consider Your Audience**
- **Definition**: Understand who your guide is for. Tailor the content's depth and complexity based on the audience's level of expertise.
- **Example**: For beginners, provide more background information and definitions, while experts might appreciate advanced tips and tricks.

### 4. **Provide Clear Context**
- **Definition**: Provide clear context on the who, what, and purpose of your prompt. This helps the AI model understand the task better.
- **Example**: "You are a doctor. Read this medical history and predict risks for the patient. January 1, 2000: Fractured right arm playing basketball. Treated with a cast. February 15, 2010: Diagnosed with hypertension. Prescribed lisinopril. September 10, 2015: Developed pneumonia. Treated with antibiotics and recovered fully. March 1, 2022: Sustained a concussion in a car accident. Admitted to the hospital and monitored for 24 hours".

### 5. **Use Specific Instructions**
- **Definition**: Use specific commands to instruct the model what you want to achieve, such as "Write", "Classify", "Summarize", "Translate", etc.
- **Example**: "Extract the name of places in the following text. Desired format: Place: <comma_separated_list_of_places>".

### 6. **Organize for Impact**
- **Definition**: Use lists or comma-separated values to organize your prompt for better impact.
- **Example**: "Task: Write an introductory message. Topic: Excited to be part of the team. My Name: Embracer. Superpowers: hugs, energy. Team: Friendly Force. Exclude words: villain, dark. Style: creative. Tone: confident, energetic. Length: 50 words".

### 7. **Iterate and Refine**
- **Definition**: AI responses can vary, so it's important to iterate and refine your prompts based on feedback.
- **Example**: "Embrace the chaos and iterate. AI is unpredictable and moody, with fluid responses that rely on your prompt structure".

### Conclusion
By following these tips, beginners can create effective prompts that yield accurate and relevant responses from AI models. Remember to keep your prompts clear, concise, and specific, and to iterate based on feedback to achieve the best results.

**Sources:**
- [A Guide to Crafting Effective Prompts for Diverse Applications]
- [7 Tips for Powerful Prompt Design]
- [General Tips for Designing Prompts]

In [147]:
import os
from tavily import TavilyClient

# Tavily API 키 설정
tavily_api_key = os.environ.get("TAVILY_API_KEY")

if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY 환경 변수가 설정되지 않았습니다.")

# Tavily 클라이언트 초기화
client = TavilyClient(api_key=tavily_api_key)

def perform_search(query, search_depth="basic", max_results=5, include_answer=True):
    """
    Tavily API를 사용하여 검색을 수행합니다.
    
    :param query: 검색할 쿼리 문자열
    :param search_depth: 검색 깊이 ('basic' 또는 'advanced')
    :param max_results: 반환할 최대 결과 수
    :return: 검색 결과 리스트
    """
    try:
        response = client.search(query=query, search_depth=search_depth, max_results=max_results, include_answer=include_answer)
        return response
    except Exception as e:
        print(f"검색 중 오류 발생: {e}")
        return []

search_query = "Elasticsearch 의 ELSER 검색에 대해 알려줘."
response = perform_search(search_query)

print(f"'{search_query}' 검색 결과:")
print(f"AI 답변: {response["answer"]}")
results = response["results"]
for idx, result in enumerate(results, 1):
    print(f"\n{idx}. {result['title']}")
    print(f"   URL: {result['url']}")
    print(f"   내용 요약: {result['content'][:300]}...")
    print(f"   내용 길이: {len(result['content'])}")

'Elasticsearch 의 ELSER 검색에 대해 알려줘.' 검색 결과:
AI 답변: Elastic Learned Sparse EncodeR (ELSER) is a retrieval model trained by Elastic that enables semantic search in Elasticsearch. It provides search results based on contextual meaning and user intent rather than exact keyword matches. ELSER is now available for production use in Elasticsearch 8.11, offering best-in-class relevance and integration with transformer models for improved search capabilities.

1. ELSER - Elastic Learned Sparse EncodeR | Machine Learning in the ...
   URL: https://www.elastic.co/guide/en/machine-learning/current/ml-nlp-elser.html
   내용 요약: Elastic Learned Sparse EncodeR - or ELSER - is a retrieval model trained by Elastic that enables you to perform semantic search to retrieve more relevant search results. This search type provides you search results based on contextual meaning and user intent, rather than exact keyword matches. ELSER...
   내용 길이: 350

2. Elastic Search 8.11: ELSER model is now GA and customers c

In [262]:
# Research 테스트 
from tavily import TavilyClient
import os
import pprint
from collections import defaultdict
import json
from typing import List, Dict, Any, Optional
from elasticsearch import Elasticsearch
from tavily import TavilyClient
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from itertools import islice
from cohere import TooManyRequestsError
import random
import asyncio
import aiohttp
from typing import List, Dict, Any
import nest_asyncio

# 이벤트 루프 중첩 허용
nest_asyncio.apply()

class AsyncTavilyClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.tavily.com/search"

    async def search(self, session, **params):
        headers = {
            "Content-Type": "application/json"
        }
        
        # API 키를 params에 추가
        params['api_key'] = self.api_key
        
        # 부울 값을 문자열로 변환
        processed_params = {
            k: str(v).lower() if isinstance(v, bool) else v
            for k, v in params.items()
        }
        
        async with session.post(self.base_url, json=processed_params, headers=headers) as response:
            content_type = response.headers.get('Content-Type', '')
            if 'application/json' in content_type:
                return await response.json()
            elif 'text/html' in content_type:
                html_content = await response.text()
                print(f"Received HTML response: {html_content[:200]}...")  # 처음 200자만 출력
                raise ValueError(f"Received HTML response instead of JSON. Status: {response.status}")
            else:
                raise ValueError(f"Unexpected content type: {content_type}")
        
class TabilyRetriever:
    def __init__(self, api_key, sub_query_generator):
        self.tavily = AsyncTavilyClient(api_key)
        self.sub_query_generator = sub_query_generator
        print('Created to Tavily! successfully')

    async def async_search(self, query, max_results=5):
        params = {
            "query": query, 
            "count": max_results - 1
        } 
        
        sub_queries = self.sub_query_generator.invoke(params)
        original_query = sub_queries["original_query"]
        sub_queries = sub_queries["sub_queries"]
        print(f"sub_queries: {sub_queries}")
        
        async with aiohttp.ClientSession() as session:
            tasks = [
                self.tavily.search(session, query=original_query, max_results=3, search_depth="basic", include_answer=True),
                *[self.tavily.search(session, query=sub_query, max_results=3, search_depth="advanced", include_answer=True) for sub_query in sub_queries]
            ]
            responses = await asyncio.gather(*tasks)

        final_results = []
        print(f"basic_response[0]: {responses[0]}")
        basic_response = responses[0]
        final_results.append({
            "id": "basic_tavily_search_0",
            'content': basic_response["answer"],
            'metadata': {'title': basic_response["results"][0]["title"], 'source': basic_response["results"][0]["url"], 'search_method': 'tavily'}
        })
        for i, advanced_response in enumerate(responses[1:], 1):
            final_results.append({
                "id": f'advanced_tavily_search_{sub_queries[i-1]}',
                'content': advanced_response["answer"],
                'metadata': {'title': advanced_response["results"][0]["title"], 'source': advanced_response["results"][0]["url"], 'search_method': 'tavily'}
            })
            
        return final_results

    def search(self, query: str, max_results: int = 5) -> List[Dict[str, Any]]:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(self.async_search(query, max_results))


class PerplexityRetriever:
    def __init__(self, api_key):
        self.api_key = api_key
        self.client = OpenAI(api_key=api_key, base_url="https://api.perplexity.ai")
        print('Created to Perplexity! successfully')

    def search(self, queries: List[str], main_topic, sub_topic) -> List[Dict[str, Any]]:
        results = [] 
        messages = [
            {
                "role": "system",
                "content": (
                    """
                    You are AI Assitant. 
                    
                    1. Objective: 
                    - Deliver concise, accurate, and comprehensive responses to user queries.
                    
                    2. Tone and Style: 
                    - Maintain a journalistic tone.
                    - Avoid moralizing or hedging language.
                    
                    3. Citations:
                    - Cite relevant search results using the format [index] at the end of sentences.
                    - Limit citations to a maximum of three results per sentence.
                    - Avoid citing irrelevant results.
                    
                    4. Markdown Formatting:
                    - Use level 2 headers (##) for main sections.
                    - Use bolding (****) for subsections.
                    - Use unordered lists for regular lists; use ordered lists only when ranking or if contextually appropriate.
                    - Use markdown code blocks for code snippets, specifying the language for syntax highlighting.
                    - Wrap all math expressions in LaTeX using double dollar signs ($$).
                    
                    5. Response Structure:
                    - Directly answer the query without unnecessary introductions.
                    - Ensure the response is self-contained and fully addresses the query.
                    
                    6. Additional Guidelines:
                    - Use italics for terms or phrases needing subtle emphasis.
                    - Maintain a clear visual hierarchy in the response.
                    - Avoid including URLs or links in the answer.
                    - Omit bibliographies at the end of answers.
                    
                    7. Handling Uncertainty:
                    - If unsure of the answer or if the premise is incorrect, explain why.
                    - If search results are unhelpful, answer based on existing knowledge.
                    """
                ),
            }
        ]
        
        for query in queries:
            content = f" I'm planning to create a lecture on the main topic '{main_topic}' and subtopic '{sub_topic}' for an audience of beginner developers. If you reference any documents or papers when providing your answer, please be sure to include the sources. answer the following question: {query}. "
            messages.append(
                {
                    "role": "user",
                    "content": content
                }
            )
        
            response = self.client.chat.completions.create(
                model="llama-3.1-sonar-large-128k-online",
                messages=messages,
                max_tokens=2048,
                temperature=0.1 
            )
            
            messages.append(
                {
                    "role": "assistant",
                    "content": response.choices[0].message.content
                }
            )
            
            results.append(
                {
                    "id": f'perplexity_search_{query}',
                    'content': response.choices[0].message.content,
                    'metadata': {'title': query, 'source': 'https://perplexity.ai', 'search_method': 'Perplexity'}
                }
            ) 
        
        return results 
        


class ElsatsticsearchRetriever: 
    def __init__(self, es_client, model_id, index, coheres, embedding):
        self.es = es_client
        self.model_id = model_id
        self.index = index
        self.coheres = coheres
        self.embedding = embedding
        print('Created to Elasticsearch! successfully')

    def hybrid_search(self, queries, max_results=5):
        results = []
        for query in queries:
            results.extend(self._hybrid_search_single(query, max_results))
        
        return results

    def _hybrid_search_single(self, query, max_results=5):
        search_query = {
            "sub_searches": [
                {
                    "query": { 
                        "multi_match": {
                            "query": query,
                            "fields": ["text", "metadata.fulltext"],
                            "type": "best_fields",
                            "tie_breaker": 0.3
                        }
                    }
                }, 
                {
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": self.model_id,
                                "model_text": query,
                                "boost": 2
                            }
                        }
                    }
                }
            ],
            "knn": {
                'field': 'text_dense_embedding',
                'query_vector': self._get_embedding(query),
                'k': 10,
                'num_candidates': 100,
            },
            "rank": {
                "rrf": {} 
            }, 
            "min_score": 15
        }
        
        response = self.es.search(
            index=self.index, 
            size=max_results,
            body = search_query
        )
        
        if not response["hits"]["hits"]:
            return []
        
        def extract_info(hit):
            source = hit["_source"]
            metadata = source.get("metadata", {})
            return {
                "id": hit["_id"],
                "index": hit["_index"],
                "score": hit["_score"],
                "content": source.get("text"),
                "metadata": {
                    "title": metadata.get("heading"),
                    "source": metadata.get("source"),
                    "search_method": "Elasticsearch(Hybrid Search)",
                },
            }
            
        hits_dict = {hit["_id"]: extract_info(hit) for hit in response["hits"]["hits"]}
    
        refined_docs = [{'id': id, 'text': hit['content']} for id, hit in hits_dict.items()]
        
        reranked_ids = [doc['id'] for doc in self._rerank_docs(query, refined_docs, len(refined_docs))]
        
        return list(islice((hits_dict[id] for id in reranked_ids if id in hits_dict), max_results))

        
    def _rerank_docs(self, query, docs, size):
        rerank_results = self._rerank_docs_call(query, docs, size)
        
        reranked_docs = []
        for rerank_result in rerank_results.results:
            index = rerank_result.index
            reranked_docs.append(docs[index])
            
        return reranked_docs
    
    def _rerank_docs_call(self, query, docs, size):
        def call(): 
            return co.rerank(
                    model="rerank-english-v3.0", 
                    query=query, 
                    documents=docs, 
                    top_n=size 
                )
        
        for _ in range(len(self.coheres)):
            co = self.coheres[random.randint(0, len(self.coheres) - 1)]
            try:
                rerank_results = call()
                
                if (rerank_results.results is None) or (len(rerank_results.results) == 0):
                    rerank_results = call()
                
                return rerank_results
            except TooManyRequestsError:
                continue  # 다음 cohere 인스턴스로 넘어감
    
    def _get_embedding(self, text):
        return self.embedding.encode(text)



class Retriever:
    def __init__(self, perplexity_retriever, elasticsearch_retriver, embedding_model, coheres):
        self.perplexity_retriever = perplexity_retriever
        self.elasticsearch_retriver = elasticsearch_retriver
        self.embedding_model = embedding_model
        self.coheres = coheres
        print('Created to Retriever! successfully')

    def _search_perplexity(self, queries: List[str], main_topic: str = "", sub_topic: str = "") -> List[Dict[str, Any]]:
        return self.perplexity_retriever.search(queries, main_topic, sub_topic)

    def _search_elasticsearch(self, queries: List[str], max_results: int) -> List[Dict[str, Any]]:
        results = self.elasticsearch_retriver.hybrid_search(queries, max_results)
        if not results:
            return []
        return results

    def _get_text_for_embedding(self, result: Dict[str, Any]) -> str:
        return result.get('content', '')

    def _compute_similarity(self, embedding1: np.ndarray, embedding2: np.ndarray) -> float:
        return cosine_similarity([embedding1], [embedding2])[0][0]

    def _process_results(self, results: List[Dict[str, Any]], similarity_threshold: float = 0.8) -> List[Dict[str, Any]]:
        processed_results = []

        for result in results:
            result_text = self._get_text_for_embedding(result)
            
            if result_text == '': 
                continue
            
            result_embedding = self.embedding_model.encode([result_text])[0]
            
            # Check if the result is similar to any previously processed result
            is_similar_to_previous = any(
                self._compute_similarity(result_embedding, processed['embedding']) > similarity_threshold
                for processed in processed_results
            )
            
            if not is_similar_to_previous:
                result['embedding'] = result_embedding
                processed_results.append(result)

        for result in processed_results:
            del result['embedding']
            
        return processed_results

    def search(self, queries: List[str], max_results: int = 10, main_topic: str = "", sub_topic: str = "") -> List[Dict[str, Any]]:
        es_results = self._search_elasticsearch(queries, max_results)
        perplexity_results = self._search_perplexity(queries, main_topic, sub_topic)
        # Combine all results
        combined_results = perplexity_results + es_results

        # Process results: remove duplicates based on embedding similarity
        processed_results = self._process_results(combined_results)
        print(f"sources: {[result["metadata"]["source"] for result in processed_results]}")
        
        results_map = {result["id"]: result for result in processed_results} 
        
        refined_results = [{'id': result['id'], 'text': result['content']} for result in processed_results]
        
        reranked_ids = [result['id'] for result in self.rerank_results(query, refined_results, len(refined_results))]
            
        return list(islice((results_map[id] for id in reranked_ids if id in results_map), max_results))
        

    def retrieve(self, queries: List[str], max_results: int = 10, main_topic: str = "", sub_topic: str = "") -> List[Dict[str, Any]]:
        return self.search(queries, max_results, main_topic, sub_topic)
    
    
    def rerank_results(self, query, docs, size):
        rerank_results = self.rerank_results_call(query, docs, size)
        
        reranked_docs = []
        for rerank_result in rerank_results.results:
            index = rerank_result.index
            reranked_docs.append(docs[index])
            
        return reranked_docs
    
    def rerank_results_call(self, query, docs, size):
        def call(): 
            return co.rerank(
                    model="rerank-english-v3.0", 
                    query=query, 
                    documents=docs, 
                    top_n=size 
                )
        
        for _ in range(len(self.coheres)):
            co = self.coheres[random.randint(0, len(self.coheres) - 1)]
            try:
                rerank_results = call()
                
                if (rerank_results.results is None) or (len(rerank_results.results) == 0):
                    rerank_results = call()
                
                return rerank_results
            except TooManyRequestsError:
                continue  # 다음 cohere 인스턴스로 넘어감

In [263]:
from elasticsearch import Elasticsearch, NotFoundError
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os 
from dotenv import load_dotenv
import cohere
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import SentenceTransformer

load_dotenv(override=True)

INDEX = "lecture-content-v4"
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
ELSER_MODEL = os.getenv("ELSER_MODEL")

COHERE_API_KEY= os.getenv("COHERE_API_KEY")
COHERE_API_KEY2 = os.getenv("COHERE_API_KEY2")
COHERE_API_KEY3 = os.getenv("COHERE_API_KEY3")
COHERE_API_KEY4 = os.getenv("COHERE_API_KEY4")
COHERE_API_KEY5 = os.getenv("COHERE_API_KEY5")

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

PER_PLEXITY_API_KEY = os.getenv("PER_PLEXITY_API_KEY")

coheres = [
    cohere.Client(COHERE_API_KEY),
    cohere.Client(COHERE_API_KEY2),
    cohere.Client(COHERE_API_KEY3),
    cohere.Client(COHERE_API_KEY4),
    cohere.Client(COHERE_API_KEY5),
]

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

elasticsearch_client = Elasticsearch(cloud_id=ELASTIC_CLOUD_ID, api_key=ELASTIC_API_KEY)

sub_query_generator_prompt = ChatPromptTemplate.from_template(
    """
    You are an analytical AI assistant. Your task is to analyze a given query, clarify its meaning, and decompose it into three manageable sub-queries. This process will help in systematically approaching complex questions.

    For the given query, follow these steps:

    1. Query Analysis:
    - Carefully read the given query and grasp its overall meaning and purpose.
    - Identify the main concepts, terms, or topics included in the query.
    - Determine the type of information or task the query is requesting.

    2. Meaning Clarification:
    - Briefly restate the main points of the query.
    - If there are any ambiguous or unclear parts, point them out and suggest possible interpretations.

    3. Sub-query Decomposition:
    - Break down the main query into {count} logical and manageable sub-queries.
    - Ensure that each sub-query addresses a specific aspect of the original query.
    - Verify that the sub-queries together cover all aspects of the original query.

    Your output should be in JSON format as follows:

    {{
    "original_query": "The full text of the original query",
    "query_clarification": "A brief clarification of the query's meaning",
    "sub_queries": [
        "First sub-query",
        "Second sub-query",
        "Third sub-query"
    ]
    }}

    Using this structure, please analyze and decompose the following query:

    Query: {query}
    """
)

llm = ChatOpenAI(model="gpt-4o")

sub_query_generator = sub_query_generator_prompt | llm | JsonOutputParser()

# es_client, model_id, index, coheres, embedding):
elasticsearch_retriver = ElsatsticsearchRetriever(es_client=elasticsearch_client, model_id=ELSER_MODEL, index=INDEX, coheres=coheres, embedding=embedding_model) 

tavily_retriever = TabilyRetriever(api_key=TAVILY_API_KEY, sub_query_generator=sub_query_generator)

perplexity_retriever = PerplexityRetriever(api_key=PER_PLEXITY_API_KEY) 

#tavily_retriever, elasticsearch_retriver, embedding_model, coheres
retriever = Retriever(perplexity_retriever=perplexity_retriever, elasticsearch_retriver=elasticsearch_retriver, embedding_model=embedding_model, coheres=coheres) 

/Users/jeongmin/PycharmProjects/tech-blog-article-summary/.env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Created to Elasticsearch! successfully
Created to Tavily! successfully
Created to Perplexity! successfully
Created to Retriever! successfully


In [261]:
# Retriever 테스트
# results = elasticsearch_retriver.hybrid_search("Elasticsearch Hybrid Search", max_results=5)
# results = tavily_retriever.search(query_refine_result['refined_queries'][0], max_results=5)
# results = retriever.retrieve(query_refine_result['refined_queries'][0], max_results=5)
# results = perplexity_retriever.search(queries, main_topic, sub_topic)
# results = elasticsearch_retriver.hybrid_search(queries)

# results = retriever.search(queries, main_topic=main_topic, sub_topic=sub_topic_title, max_results=5)

from IPython.display import display, Markdown

main_topic = sub_queries_refine_result["main_topic"]
sub_topic = sub_queries_refine_result["sub_topics"][0]
sub_topic_title = sub_topic["title"]
queries = sub_topic["revised_search_queries"]

results = retriever.search(queries, main_topic=main_topic, sub_topic=sub_topic_title, max_results=5)

if results:
    display(Markdown(f"{results[0]['content']}"))
    display(Markdown(f"{results[1]['content']}"))
    display(Markdown(f"{results[2]['content']}"))

/var/folders/r4/w6gk0qbd6bd_sf7xj6nwdnxc0000gn/T/ipykernel_74611/862914172.py:241: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  response = self.es.search(


sources: ['https://perplexity.ai', 'https://perplexity.ai', 'https://perplexity.ai']


## Components of a Prompt in AI

When creating a lecture on "Basic Prompting" with a subtopic of "Understanding Prompt Structure" for beginner developers, it's essential to cover the fundamental components of a prompt in AI. These components are crucial for crafting effective prompts that elicit desired responses from AI models like ChatGPT.

### 1. **Clear Objective**
The prompt should clearly state the objective or task that the AI needs to perform. This helps the model understand what is expected of it and ensures the response is relevant and accurate.

### 2. **Specific Instructions**
Providing specific instructions within the prompt helps guide the AI model to produce the desired output. This can include details about the format, tone, and any specific data points or examples that should be included.

### 3. **Context**
Including context in the prompt is vital for the AI model to understand the background and relevance of the task. This can involve providing examples, definitions, or any other relevant information that might help the model generate a more accurate response.

### 4. **Constraints**
Specifying constraints or limitations in the prompt can help the AI model stay focused and avoid generating irrelevant information. This can include constraints on the length of the response, the tone, or specific topics to avoid.

### 5. **Examples**
Providing examples within the prompt can help the AI model understand the expected output better. Examples can serve as a guide for the model to follow and ensure that the response is in line with the desired format and content.

### 6. **Feedback Mechanism**
Incorporating a feedback mechanism within the prompt allows for iterative refinement. This can involve asking the AI model to provide suggestions for improving the prompt or to refine the response based on feedback.

### 7. **Repetition of Instructions**
Repeating instructions at the beginning and end of the prompt can help reinforce the task and ensure that the AI model does not lose focus. This technique can improve the accuracy and relevance of the response.

By understanding and incorporating these components into prompts, developers can create more effective and efficient interactions with AI models, leading to better outcomes in various applications.

---

**References:** https://www.reddit.com/r/ChatGPT/comments/14d7pfz/become_god_like_prompt_engineer_with_this_one/ https://community.openai.com/t/prompt-engineering-for-rag/621495 https://community.openai.com/t/openais-dec-17th-2023-prompt-engineering-guide/562526 https://www.semrush.com/blog/chatgpt-prompts/

## Prompt Instructions, Context, and Examples Explained

When structuring a prompt for language models, three key components are essential: instructions, context, and examples. These elements work together to guide the model in producing the desired response.

### **Instructions**
Instructions are specific tasks or commands that guide the model's response. They should be clear and unambiguous to ensure the model understands what is expected of it. For example, an instruction-based prompt might be: "Play the role of an experienced Python developer and help me write code".

### **Context**
Context provides additional information that helps the model understand the background and relevance of the task. This can include examples, definitions, or any other relevant details that might help the model generate a more accurate response. For instance, a contextual prompt might be: "Take this research document as the context and please answer the questions based on that".

### **Examples**
Examples are used to illustrate the expected output and help the model understand the format and content required. They can be particularly useful in ensuring the model's response is specific and accurate. For example, in a text classification task, providing examples can help the model understand the correct format for the output: "Classify the text into neutral, negative or positive. Text: I think the food was okay. Sentiment: neutral".

By combining these components effectively, developers can create well-structured prompts that guide language models to produce accurate and relevant responses.

---

**References:**

## Effective Prompt Design for Beginners

Effective prompt design is crucial for beginners to maximize the potential of AI tools. Here are some key tips and strategies to help beginners craft effective prompts:

### 1. **Start Simple**
Begin with simple prompts and gradually add complexity as needed. This iterative process helps in refining the prompt to achieve better results.

### 2. **Clear Instructions**
Use clear and specific instructions to guide the model. For example, use commands like "Write," "Classify," "Summarize," or "Translate" to clearly define the task.

### 3. **Provide Context**
Include relevant context to help the model understand the background and relevance of the task. This can involve providing examples, definitions, or other relevant information.

### 4. **Be Specific and Concise**
Be very specific about the instruction and task you want the model to perform. Avoid overly complex or vague prompts. The more descriptive and detailed the prompt is, the better the results.

### 5. **Use Examples**
Providing examples in the prompt is very effective in getting the desired output in specific formats. Examples help the model understand the expected output better.

### 6. **Iterate and Refine**
Experiment with different variations of prompts and iterate based on model responses to fine-tune instructions. This helps to achieve the desired outcomes and ensures the model is aligned with user intent.

### 7. **Use Lists and Clear Separators**
Organize your prompts using lists or clear separators like "###" to separate instructions and context. This helps the model to understand the structure and focus on the task at hand.

### 8. **Consider Your Audience**
Understand who your guide is for and tailor the content accordingly. Beginners may need more background information and definitions, while experts might appreciate advanced tips and tricks.

### 9. **Avoid Impreciseness**
Be specific and direct in your prompts. Avoid vague descriptions that might confuse the model. For example, instead of "Explain the concept of prompt engineering," use "Use 2-3 sentences to explain the concept of prompt engineering to a high school student".

By following these guidelines, beginners can create effective prompts that guide AI models to produce accurate and relevant responses.

---

**References:**

In [251]:

display(Markdown(f"{results[0]['content']}"))
display(Markdown(f"{results[1]['content']}"))
display(Markdown(f"{results[2]['content']}"))

## Components of a Prompt in AI

When designing prompts for AI models, several key components are essential to ensure clarity, effectiveness, and accuracy. These components include:

### 1. **Clear Instructions**
- **Purpose**: Clearly define the task the model needs to perform.
- **Example**: "Your task is to verify if a statement is supported by a specific quote from the following set of snippets".

### 2. **Context**
- **Purpose**: Provide relevant background information or context that the model needs to understand the task.
- **Example**: "Answer the QUESTION below using the DOCUMENT below as context".

### 3. **Examples**
- **Purpose**: Include examples to illustrate the task and help the model understand the expected output.
- **Example**: "Provide a Python code snippet for parsing JSON data and printing specific values".

### 4. **Meta Prompts**
- **Purpose**: Directives that correct or guide the model's behavior to produce desired responses.
- **Example**: "You must be kind and seek common ground. Try not to repeat your responses".

### 5. **Conditional Statements**
- **Purpose**: Explain what the model should do in specific circumstances.
- **Example**: "If the DOCUMENT doesn’t contain the facts to answer the QUESTION return {NONE}".

### 6. **Repetition of Instructions**
- **Purpose**: Repeating instructions at the beginning and end of the prompt can help maintain focus.
- **Example**: "Remember, from above, your instructions are as follows: ${INSTRUCTIONS}".

By incorporating these components, developers can create well-structured prompts that guide AI models to produce accurate and relevant responses.

## How to Structure a Prompt for Language Models

Structuring a prompt for language models involves several key steps to ensure clarity, effectiveness, and accuracy. Here are the main components and best practices to consider:

### 1. **Clear Instructions**
- **Purpose**: Clearly define the task the model needs to perform.
- **Example**: "Your task is to verify if a statement is supported by a specific quote from the following set of snippets".

### 2. **Context**
- **Purpose**: Provide relevant background information or context that the model needs to understand the task.
- **Example**: "Answer the QUESTION below using the DOCUMENT below as context".

### 3. **Examples**
- **Purpose**: Include examples to illustrate the task and help the model understand the expected output.
- **Example**: "Provide a Python code snippet for parsing JSON data and printing specific values".

### 4. **Meta Prompts**
- **Purpose**: Directives that correct or guide the model's behavior to produce desired responses.
- **Example**: "You must be kind and seek common ground. Try not to repeat your responses".

### 5. **Conditional Statements**
- **Purpose**: Explain what the model should do in specific circumstances.
- **Example**: "If the DOCUMENT doesn’t contain the facts to answer the QUESTION return {NONE}".

### 6. **Repetition of Instructions**
- **Purpose**: Repeating instructions at the beginning and end of the prompt can help maintain focus.
- **Example**: "Remember, from above, your instructions are as follows: ${INSTRUCTIONS}".

### 7. **Prefixes and Delimiters**
- **Purpose**: Use prefixes or delimiters to clearly separate different parts of the prompt.
- **Example**: "TASK:, CLASSES:, OBJECTS:".

### 8. **Iterative Refinement**
- **Purpose**: Experiment with different variations of prompts and iterate based on model responses to fine-tune instructions.
- **Example**: "Experiment with different variations of prompts and iterate based on model responses to fine-tune instructions".

### 9. **Task Definition**
- **Purpose**: Clearly define the task or objective in the prompt and specify the format or structure expected in the response.
- **Example**: "Clearly define the task or objective in the prompt and specify the format or structure expected in the response".

### 10. **Prompt Length**
- **Purpose**: Keep prompts concise and focused while providing sufficient information.
- **Example**: "Keep prompts concise and focused while providing sufficient information".

By incorporating these components and following these best practices, developers can create well-structured prompts that guide language models to produce accurate and relevant responses.

## Prompt Instructions, Context, and Examples Explained

When crafting prompts for AI models, it is crucial to include clear instructions, provide relevant context, and offer examples to guide the model's response. Here is a detailed explanation of each component:

### 1. **Prompt Instructions**
- **Purpose**: Clearly define the task the model needs to perform.
- **Example**: "Your task is to verify if a statement is supported by a specific quote from the following set of snippets".

### 2. **Context**
- **Purpose**: Provide relevant background information or context that the model needs to understand the task.
- **Example**: "Answer the QUESTION below using the DOCUMENT below as context".

### 3. **Examples**
- **Purpose**: Include examples to illustrate the task and help the model understand the expected output.
- **Example**: "Provide a Python code snippet for parsing JSON data and printing specific values".

### Practical Application
- **Step-by-Step Instructions**: Break down complex tasks into simpler prompts, where each prompt builds upon the previous one’s response. This technique is known as chaining prompts.
- **Guided Reasoning**: Structure prompts to lead the AI through a step-by-step reasoning process, often used for complex analytical tasks.
- **Specificity**: Be clear and specific about what the AI should generate. For example, "Explain how photosynthesis works in simple terms for a middle school science project, focusing on the steps involved and its importance to the ecosystem".

### Contextual Considerations
- **Audience and Tone**: Specify the audience and tone to guide the AI's response. For instance, "Give me ideas for a best man’s speech that is funny and heartwarming but appropriate for a family audience".
- **Feedback and Iteration**: Provide feedback on the AI's responses and iterate on the prompts to improve accuracy and relevance. This process is part of the refinement cycle.

### Example Prompts
- **AI Art Prompts**: For AI art generators, describe the content, subject, and relevant details. For example, "An illustration of a red owl with bright blue eyes".
- **Text-Based Prompts**: For text-based AI, use clear objectives and break down complex questions. For example, "Create a script for a scene set in a mystical forest outside of New York, where two rival magicians, influenced by the style of Shakespearean dialogue, confront each other to resolve an ancient feud".

By incorporating these components and following these best practices, developers can create well-structured prompts that guide AI models to produce accurate and relevant responses.

In [116]:
from IPython.display import display, Markdown 

display(Markdown(results[0]['content']))

The core concepts of Elasticsearch Hybrid Search include indexing, searching, filtering, and aggregations. These concepts are fundamental to building a fast and accurate product search engine, as highlighted in a comprehensive guide on Elasticsearch Hybrid Search.

In [21]:
# Answering Claude 3.5 sonnet 
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers.string import StrOutputParser
from IPython.display import display, Markdown

model = ChatAnthropic(model='claude-3-sonnet-20240229')

answer = answering_questions_prompt | model | StrOutputParser() 

params = {
    "question": query_refine_result['refined_queries'][0], 
    "research_input": "\n\n".join([result['content'] for result in results])
}

answer_result = answer.invoke(params)

display(Markdown(answer_result))

NameError: name 'query_refine_result' is not defined

In [41]:
# Research 테스트 
from tavily import TavilyClient
import os
import pprint

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

research_results = [] 
for q in query_writer_result["generated_query"][0:5]:
    response = tavily.search(q, max_results=3)
    for r in response["results"]:
        research_results.append(r['content'])

pprint.pprint(research_results)


["This article quickly explains Elasticsearch's full-text search techniques "
 'and parameters. For more detailed information and the correct combination of '
 'techniques and parameters, refer to the ...',
 'Its strength in handling structured and unstructured data makes it ideal for '
 'applications requiring diverse data types and complex search queries. To '
 'optimize PostgreSQL Full-Text Search, consider fine-tuning indexing '
 'parameters based on query patterns and dataset characteristics. This '
 'approach can significantly enhance search performance and ...',
 'Elasticsearch offers a rich set of querying capabilities to search and '
 'retrieve data from indexed documents. Queries can range from simple searches '
 'for specific terms to complex aggregations and analytics. Understanding how '
 'to construct and use queries is essential for harnessing the full potential '
 'of Elasticsearch. Prerequisites.',
 'So if you want to build and configure a high-performing Elasticsearch

In [42]:
# Research Filtering 테스트
from langchain.globals import set_llm_cache
from langchain_core.output_parsers.json import JsonOutputParser
import pprint

set_llm_cache(None)

research_filter = research_filter_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | JsonOutputParser()

research_topic = topic_generated_result["generated_topics"]

params = {
    "research_topic": research_topic, 
    "search_query": "",
    "search_results": [],
    "previous_research_summary": ""
}

research_passed_result = []

research_grouped_results = [research_results[i:i+3] for i in range(0, len(research_results), 3)]

for rgs in research_grouped_results:
    params["search_results"] = rgs
    is_filtered_list = research_filter.invoke(params)

    for fl in is_filtered_list["results"]:
        if fl["is_passed"]:
            research_passed_result.append(
                {
                    "is_passed": fl["is_passed"],
                    "reason": fl["reason"], 
                    "research": rgs
                }
            )

pprint.pprint(research_passed_result)

/var/folders/r4/w6gk0qbd6bd_sf7xj6nwdnxc0000gn/T/ipykernel_42969/2860871528.py:21: RuntimeWarning: assigning None to unbound local 'i'
  research_grouped_results = [research_results[i:i+3] for i in range(0, len(research_results), 3)]
/var/folders/r4/w6gk0qbd6bd_sf7xj6nwdnxc0000gn/T/ipykernel_42969/2860871528.py:21: RuntimeWarning: assigning None to unbound local 'i'
  research_grouped_results = [research_results[i:i+3] for i in range(0, len(research_results), 3)]
/var/folders/r4/w6gk0qbd6bd_sf7xj6nwdnxc0000gn/T/ipykernel_42969/2860871528.py:21: RuntimeWarning: assigning None to unbound local 'i'
  research_grouped_results = [research_results[i:i+3] for i in range(0, len(research_results), 3)]


[{'is_passed': True,
  'reason': 'The information about optimizing hardware and memory for '
            'Elasticsearch is directly relevant to performance optimization '
            'strategies, providing practical insights that align with the '
            'research topic.',
  'research': ['So if you want to build and configure a high-performing '
               'Elasticsearch, here are the most important points to focus on. '
               '1. Hardware. You can do all the optimization possible, but if '
               "you don't have enough hardware, you'll still fall short on "
               'performance.',
               'Search filters. Effective use of filters in Elasticsearch '
               'queries can improve search performance dramatically as the '
               'filter clauses are 1) cached, and 2) able to reduce the target '
               'documents to be searched in the query clause. Wildcard '
               'queries. Avoid wildcard, especially leading wildcard que

In [67]:
from IPython.display import display, Markdown


# for research_passed in research_passed_result:
#     for r in research_passed["research"]:
        # display(Markdown(r))
        

research_passed_str = ""
research_passed_set = set()
for i, research_passed in enumerate(research_passed_result, 1):
    research_str = ""
    for r in research_passed["research"]:
        research_str += f"{r}\n"
    if research_str not in research_passed_set:
        research_passed_set.add(research_str)
        research_passed_str += f"Research {i}:\n{research_str}\n"

display(Markdown(research_passed_str))

Research 1:
So if you want to build and configure a high-performing Elasticsearch, here are the most important points to focus on. 1. Hardware. You can do all the optimization possible, but if you don't have enough hardware, you'll still fall short on performance.
Search filters. Effective use of filters in Elasticsearch queries can improve search performance dramatically as the filter clauses are 1) cached, and 2) able to reduce the target documents to be searched in the query clause. Wildcard queries. Avoid wildcard, especially leading wildcard queries, which causes the entire Elasticsearch index to be ...
Give memory to the filesystem cache edit. Give memory to the filesystem cache. Elasticsearch heavily relies on the filesystem cache in order to make search fast. In general, you should make sure that at least half the available memory goes to the filesystem cache so that Elasticsearch can keep hot regions of the index in physical memory.

Research 4:
Instead of searching the text directly, Elasticsearch searches the index, enabling lightning-fast search responses even with large datasets. By using distributed inverted indices, Elasticsearch efficiently identifies the best matches for full-text searches, making it ideal for applications where real-time search is critical.
Recent research has shown that combinations of semantic and classic keyword searches often outperform these separate search options. Elasticsearch provides Hybrid Retrieval to run both searches in parallel to get the best of both options. My previous posts describe semantic searches: IBM provides the new Elasticsearch capabilities in the new ...
We'll identify key metrics that you need to monitor to maintain the health and performance of your Elasticsearch cluster. To learn more about Elasticsearch open source monitoring tools, check out part 3 of this series. You can learn how to monitor Elasticsearch with Sematext in part 4.

Research 5:
In conclusion, hybrid search in Elasticsearch is a powerful technique for combining the strengths of different search algorithms to achieve better search accuracy and efficiency.
RRF Implementation
To enable a combined search that returns results from both full-text and vector search methods, the full-text search logic used earlier in the handle_search() function has to be brought back. If one of these methods matches your needs then you don't need anything else, but in many cases each method of searching returns valuable results that the other method would miss, so the best option is to offer a combined result set.
 Here is the version of handle_search() that implements the hybrid search strategy:
With this version the best results from each search method are combined. Consider the following example, which has query and knn sections to request full-text and vector searches respectively, and a rrf section that combines them into a single result list.
 To implement a hybrid search strategy the search() method must receive both the query and knn arguments, each requesting a separate query.
Armed with all the vector search knowledge learned in the first article, the second article: How to Set Up Vector Search in OpenSearch, guided you through the meanders of how to set up vector search in OpenSearch using either the k-NN plugin or the new Neural Search plugin that was recently made generally available in 2.9.
 Basically, Convex Combination, also called Linear Combination, seeks to combine the normalized score of lexical search results and semantic search results with respective weights and (where 0 ,), such that:
CC can be seen as a weighted average of the lexical and semantic scores, but in contrast to OpenSearch, the weights are not expressed as percentages, i.e., they do not need to add up to 100%. RRF, however, doesn’t require any score calibration or normalization and simply scores the documents according to their rank in the result set, using the following formula, where k is an arbitrary constant meant to adjust the importance of lowly ranked documents:
Both CC and RRF have their pros and cons as highlighted in Table 1, below:
Table 1: Pros and cons of CC and RRF
 Operations
Elasticsearch Elasticsearch Hybrid Search
By Opster Expert Team - Valentin Crettaz
Updated: Oct 5, 2023
Quick links
Overview
This article is the last one in a series of five that dives into the intricacies of vector search (aka semantic search) and how it is implemented in OpenSearch and Elasticsearch.
 Such a hybrid search query is shown below:
As we can see above, a hybrid search query is simply a combination of a lexical search query (e.g., a `match` query) located in the `query` section and a vector search query specified in the `knn` section.

Research 7:
Elasticsearch, with its focus on search and retrieval, shines in use cases like log analytics, monitoring, and recommendation systems. Its real-time indexing and search capabilities make it a go ...
ElasticSearch bursted on the database scene in 2010 and has now become a staple of many IT teams' workflow. It has especially revolutionized data intensive tasks like ecommerce product search, and real-time log analysis. Read on to learn what ElasticSearch is, use cases, and pros and cons.
Armed with all the vector search knowledge learned in the first article, the second article: How to Set Up Vector Search in OpenSearch, guided you through the meanders of how to set up vector search in OpenSearch using either the k-NN plugin or the new Neural Search plugin that was recently made generally available in 2.9.
 Basically, Convex Combination, also called Linear Combination, seeks to combine the normalized score of lexical search results and semantic search results with respective weights and (where 0 ,), such that:
CC can be seen as a weighted average of the lexical and semantic scores, but in contrast to OpenSearch, the weights are not expressed as percentages, i.e., they do not need to add up to 100%. RRF, however, doesn’t require any score calibration or normalization and simply scores the documents according to their rank in the result set, using the following formula, where k is an arbitrary constant meant to adjust the importance of lowly ranked documents:
Both CC and RRF have their pros and cons as highlighted in Table 1, below:
Table 1: Pros and cons of CC and RRF
 Operations
Elasticsearch Elasticsearch Hybrid Search
By Opster Expert Team - Valentin Crettaz
Updated: Oct 5, 2023
Quick links
Overview
This article is the last one in a series of five that dives into the intricacies of vector search (aka semantic search) and how it is implemented in OpenSearch and Elasticsearch.
 Such a hybrid search query is shown below:
As we can see above, a hybrid search query is simply a combination of a lexical search query (e.g., a `match` query) located in the `query` section and a vector search query specified in the `knn` section.



In [69]:

import pprint
from langchain_core.output_parsers.string import StrOutputParser
from IPython.display import display, Markdown
from langchain_openai import OpenAI


# ChatOpenAI 모델 설정
llm = ChatOpenAI(model="gpt-4o-2024-08-06", temperature=0, max_tokens=16384)

information_extract_chain = information_extract_prompt | llm | StrOutputParser()

# 테스트를 위한 입력 파라미터
params = {
    "research_topic": research_topic,
    "filtered_research_results": research_passed_str,
    "current_research_summary": ""
}

extracted_information = information_extract_chain.invoke(params)

display(Markdown(extracted_information))


### Comprehensive Summary and Educational Value

The research on Elasticsearch hybrid search techniques provides valuable insights into optimizing search performance by combining full-text and structured queries. Key educational points include the importance of hardware resources, effective use of search filters, and the role of filesystem cache in enhancing search speed. Additionally, the research highlights the benefits of hybrid retrieval, which combines semantic and classic keyword searches to improve accuracy and efficiency. This information is crucial for developers aiming to implement high-performance search solutions in real-time applications.

For developer education, these insights can be used to teach the importance of resource allocation, query optimization, and the integration of different search methodologies. Practical applications include setting up Elasticsearch for log analytics, monitoring, and recommendation systems, where real-time search capabilities are essential. Understanding these concepts will enable developers to build robust search solutions that leverage the strengths of both full-text and vector search methods.

### Core Concepts and Technologies

- **Elasticsearch Hybrid Search**: Combines full-text and vector search methods to improve search accuracy and efficiency. (Importance: 5)
  - Related code example:
    ```json
    {
      "query": {
        "match": {
          "field": "value"
        }
      },
      "knn": {
        "field": "vector_field",
        "query_vector": [0.1, 0.2, 0.3],
        "k": 10
      }
    }
    ```
  - Real-world application: Used in e-commerce platforms to enhance product search by combining keyword and semantic search results.

### Development Methodologies and Best Practices

- **Search Filter Optimization**: Use cached filter clauses to improve search performance. (Relevance: 4)
  - Implementation steps:
    1. Identify frequently used filters.
    2. Ensure filters are cached by Elasticsearch.
    3. Avoid using wildcard queries, especially leading wildcards.

### Performance and Optimization Techniques

- **Filesystem Cache Utilization**: Allocate memory to the filesystem cache to speed up search operations. (Effectiveness: 4)
  - Benchmark results: Improved search response times by up to 30% in high-load scenarios.
  - Optimization code example:
    ```bash
    # Before
    ES_HEAP_SIZE=4g

    # After
    ES_HEAP_SIZE=2g
    # Allocate remaining memory to filesystem cache
    ```

### Security and Stability Considerations

- **Cluster Health Monitoring**: Regularly monitor key metrics to maintain Elasticsearch cluster stability. (Importance: 4)
  - Countermeasures: Use tools like Sematext for monitoring.
  - Related code patterns or libraries: Implement alerting mechanisms for critical metrics.

### Considerations, Applicability, and Limitations

- **Considerations when Applying**:
  - **Hybrid Search**: Requires careful tuning of weights for combining search results.
- **When to Apply**:
  - **Hybrid Search**: Effective in scenarios requiring both keyword and semantic search, such as recommendation systems.
- **When Not to Apply**:
  - **Hybrid Search**: May be overkill for simple search applications with limited data.

### Advantages and Disadvantages

- **Advantages**:
  - **Hybrid Search**: Combines strengths of different search algorithms, improving accuracy.
- **Disadvantages**:
  - **Hybrid Search**: Increased complexity in query formulation and result interpretation.

### Latest Trends and Future Outlook

- **Trend**: Increasing use of vector search in combination with traditional search methods. (Impact: 4)
  - Related technologies or frameworks: OpenSearch, k-NN plugin, Neural Search plugin.
  - Suggested learning roadmap:
    1. Understand basic Elasticsearch operations.
    2. Learn about vector search and its applications.
    3. Explore hybrid search implementations.

### Additional Learning Resources

- Documentation and tutorials: [Elasticsearch Documentation](https://www.elastic.co/guide/en/elasticsearch/reference/current/index.html)
- Video lectures: [Elasticsearch YouTube Channel](https://www.youtube.com/user/elasticsearch)
- Related communities and forums: [Discuss Elasticsearch](https://discuss.elastic.co/)

### Areas Needing Further Research and Verification

- **Areas requiring additional investigation**: Impact of different hardware configurations on hybrid search performance.
- **Information needing verification and reasons**: Effectiveness of new Neural Search plugin in real-world applications, as it is a relatively new feature.

In [77]:
from IPython.display import display, Markdown

# ChatOpenAI 모델 설정
llm = ChatOpenAI(model="gpt-4o-2024-08-06", temperature=0, max_tokens=16384)

# 체인 구성
research_report_chain = research_report_prompt | llm | StrOutputParser()

# 테스트를 위한 입력 파라미터
params = {
    "research_topic": research_topic,
    "extracted_information": extracted_information,
    "additional_research_results": ""
}

init_research_report = research_report_chain.invoke(params)

display(Markdown(init_research_report))

# Report Title: Advanced Techniques and Performance Optimization in Elasticsearch Hybrid Search

## 1. Overview

This report delves into the advanced techniques and performance optimization strategies for Elasticsearch hybrid search, focusing on combining full-text search with structured queries. The purpose is to provide developers with a comprehensive understanding of how to implement and optimize hybrid search solutions effectively. The report is structured to cover technical analysis, implementation guides, code examples, and future trends.

### Main Learning Objectives:
- Understand the core concepts and technologies behind Elasticsearch hybrid search.
- Learn best practices for performance optimization and resource allocation.
- Explore real-world applications and scenarios where hybrid search is beneficial.
- Gain insights into the latest trends and future directions in search technologies.

## 2. Technical Analysis and Implementation Guide

### 2.1 Elasticsearch Hybrid Search

#### 2.1.1 Context and Problem
Traditional search methods often struggle with balancing the precision of structured queries and the flexibility of full-text search. This limitation is particularly evident in applications requiring both keyword and semantic search capabilities, such as e-commerce platforms and recommendation systems.

#### 2.1.2 Definition and Key Features
Elasticsearch hybrid search combines full-text and vector search methods to enhance search accuracy and efficiency. Key features include:
- **Full-Text Search**: Allows for natural language queries.
- **Vector Search**: Utilizes machine learning models to understand semantic meaning.

#### 2.1.3 Solution
Hybrid search addresses the limitations of traditional search by integrating semantic understanding with keyword precision. This approach improves search relevance and user satisfaction by delivering more accurate results.

#### 2.1.4 Importance and Real-World Application Areas
Hybrid search is crucial for applications requiring nuanced search capabilities, such as:
- **E-commerce**: Enhancing product search by combining keyword and semantic results.
- **Recommendation Systems**: Providing personalized content suggestions.

#### 2.1.5 Relationship with Related Technologies/Concepts
Hybrid search interacts with technologies like machine learning and natural language processing (NLP) to enhance semantic understanding. It complements traditional search methods by providing a more comprehensive search experience.

#### 2.1.6 Pros and Cons Analysis
- **Pros**: 
  - Improved search accuracy and relevance.
  - Enhanced user experience through semantic understanding.
- **Cons**: 
  - Increased complexity in query formulation.
  - Higher resource requirements for processing.

#### 2.1.7 Issues and Considerations
Potential issues include the need for careful tuning of search weights and resource allocation. Ignoring these can lead to suboptimal performance and increased costs.

#### 2.1.8 When to Use
Hybrid search is effective in scenarios requiring both keyword and semantic search, such as:
- **Complex Data Sets**: Where nuanced understanding is necessary.
- **Real-Time Applications**: Where quick and accurate results are crucial.

#### 2.1.9 When Not to Use
Avoid hybrid search in simple applications with limited data, where the complexity and resource demands may not be justified.

#### 2.1.10 Development History and Latest Trends
Hybrid search has evolved with advancements in machine learning and NLP. Current trends include the integration of vector search and the use of plugins like k-NN and Neural Search.

#### 2.1.11 Use Cases
1) **Case 1**: An e-commerce platform using hybrid search to improve product recommendations, resulting in increased sales and customer satisfaction.
2) **Case 2**: A media streaming service implementing hybrid search for personalized content delivery, enhancing user engagement.

### 2.2 Performance Optimization Strategies

#### 2.2.1 Context and Problem
Elasticsearch performance can degrade under high load, affecting search speed and user experience. Optimizing performance is essential for maintaining efficient operations.

#### 2.2.2 Definition and Key Features
Performance optimization involves techniques like search filter optimization and filesystem cache utilization to enhance search speed and efficiency.

#### 2.2.3 Solution
By caching frequently used filters and allocating memory to the filesystem cache, search operations become faster and more efficient, reducing response times.

#### 2.2.4 Importance and Real-World Application Areas
Optimized performance is vital for applications with high search volumes, such as:
- **Log Analytics**: Where quick data retrieval is necessary.
- **Monitoring Systems**: Requiring real-time data processing.

#### 2.2.5 Relationship with Related Technologies/Concepts
Performance optimization is related to resource management and system architecture, ensuring efficient use of hardware and software resources.

#### 2.2.6 Pros and Cons Analysis
- **Pros**: 
  - Reduced search response times.
  - Improved system efficiency and user satisfaction.
- **Cons**: 
  - Requires careful resource management.
  - Potential for increased complexity in system configuration.

#### 2.2.7 Issues and Considerations
Considerations include ensuring adequate memory allocation and avoiding wildcard queries, which can degrade performance.

#### 2.2.8 When to Use
Use performance optimization in high-load scenarios where quick search responses are critical.

#### 2.2.9 When Not to Use
Avoid extensive optimization in low-load environments where the benefits may not justify the effort.

#### 2.2.10 Development History and Latest Trends
Performance optimization techniques have evolved with advancements in hardware and software, focusing on efficient resource utilization and system scalability.

#### 2.2.11 Use Cases
1) **Case 1**: A financial institution optimizing Elasticsearch for real-time fraud detection, improving response times and accuracy.
2) **Case 2**: A news aggregator using performance optimization to deliver timely and relevant content to users.

## 3. Code Examples and Practical Exercises

### Code Example 1: Basic Hybrid Search Query
```json
{
  "query": {
    "match": {
      "field": "value"
    }
  },
  "knn": {
    "field": "vector_field",
    "query_vector": [0.1, 0.2, 0.3],
    "k": 10
  }
}
```
**Explanation**: This example demonstrates a basic hybrid search query combining full-text and vector search.

### Code Example 2: Search Filter Optimization
```json
{
  "query": {
    "bool": {
      "filter": [
        {
          "term": {
            "status": "active"
          }
        }
      ]
    }
  }
}
```
**Explanation**: This example shows how to use cached filter clauses to improve search performance.

### Code Example 3: Filesystem Cache Configuration
```bash
# Before
ES_HEAP_SIZE=4g

# After
ES_HEAP_SIZE=2g
# Allocate remaining memory to filesystem cache
```
**Explanation**: This example illustrates how to configure Elasticsearch to utilize filesystem cache for better performance.

### Practical Project Ideas
- **Beginner**: Set up a basic Elasticsearch instance and perform simple full-text searches.
- **Intermediate**: Implement a hybrid search solution for a small e-commerce dataset.
- **Advanced**: Optimize an Elasticsearch cluster for high-load scenarios, focusing on performance and resource management.

## 4. Latest Trends and Future Outlook

### Current Industry Trends
- Increasing integration of vector search with traditional methods.
- Growing emphasis on real-time search capabilities and performance optimization.

### Predicted Technological Developments
- Enhanced machine learning models for better semantic understanding.
- Improved plugins and tools for easier hybrid search implementation.

### Impact on Developer Education and the Tech Industry
These trends will necessitate updated educational content focusing on hybrid search techniques and performance optimization, preparing developers for future challenges and opportunities.

## 5. Conclusion and Future Research Directions

### Key Findings
- Hybrid search offers significant advantages in accuracy and efficiency for complex search applications.
- Performance optimization is crucial for maintaining efficient search operations under high load.

### Conclusions
The research highlights the importance of integrating full-text and vector search methods to enhance search capabilities. Developers must focus on resource management and optimization to achieve the best results.

### Future Research Directions
Further research is needed to explore the impact of different hardware configurations on hybrid search performance and the effectiveness of new plugins like Neural Search in real-world applications.

In [71]:
from IPython.display import display, Markdown
from langchain_core.output_parsers.json import JsonOutputParser


# ChatOpenAI 모델 설정
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 체인 구성
research_filter_chain = research_filter_chain_prompt | llm | StrOutputParser()

# 테스트를 위한 입력 파라미터
params = {
    "research_report": init_research_report,
    "research_topic": research_topic,
    "target_audience": "Junior Software Deveolper"
}

filter_chain_result = research_filter_chain.invoke(params)

display(Markdown(filter_chain_result))

```json
{
    "evaluation_results": [
        {
            "criterion": "Credibility",
            "result": "Fail",
            "explanation": "The report does not cite any peer-reviewed journals, recognized industry publications, or other authoritative references.",
            "improvement_suggestion": "Include citations from peer-reviewed journals, industry whitepapers, or authoritative sources such as Elasticsearch's official documentation."
        },
        {
            "criterion": "Relevance",
            "result": "Pass",
            "explanation": "The information provided is directly related to the research topic, covering advanced techniques and performance optimization in Elasticsearch hybrid search.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Timeliness",
            "result": "Fail",
            "explanation": "The report does not provide publication dates or last updated information for key sources, making it difficult to assess the timeliness of the information.",
            "improvement_suggestion": "Include publication dates or last updated information for key sources to ensure the information is up-to-date."
        },
        {
            "criterion": "Accuracy",
            "result": "Fail",
            "explanation": "The report lacks citations for specific data points or claims, making it difficult to verify their accuracy.",
            "improvement_suggestion": "Provide citations for specific data points or claims to allow for verification of accuracy."
        },
        {
            "criterion": "Depth",
            "result": "Pass",
            "explanation": "The report covers the topic in sufficient depth, providing detailed explanations and comprehensive coverage of subtopics.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Detail",
            "result": "Pass",
            "explanation": "The report provides adequate details, breaking down complex concepts and providing examples and detailed explanations.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Objectivity",
            "result": "Pass",
            "explanation": "The report presents information in an unbiased and objective manner, balancing pros and cons and acknowledging alternative viewpoints.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Technical_Level_Appropriateness",
            "result": "Pass",
            "explanation": "The technical level of the information is suitable for junior software developers, with explanations and examples that align with their expertise level.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Practical_Examples",
            "result": "Pass",
            "explanation": "The report includes applicable examples and use cases, such as code examples and practical project ideas.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Lecture-worthy",
            "result": "Pass",
            "explanation": "The content is well-structured, with a clear logical flow and suitable for creating educational lectures.",
            "improvement_suggestion": null
        },
        {
            "criterion": "Accessible",
            "result": "Pass",
            "explanation": "The content is accessible and understandable for the target audience, with clear explanations and use of examples to simplify complex ideas.",
            "improvement_suggestion": null
        }
    ],
    "final_verdict": "Fail",
    "overall_assessment": "The research report provides a comprehensive overview of advanced techniques and performance optimization in Elasticsearch hybrid search, with detailed explanations, practical examples, and a clear structure. However, it falls short in terms of credibility, timeliness, and accuracy due to the lack of authoritative sources, publication dates, and citations for specific data points. To improve the report, it is essential to include citations from peer-reviewed journals, industry whitepapers, or authoritative sources, provide publication dates or last updated information for key sources, and ensure that specific data points or claims are verifiable. Once these improvements are made, the report will meet all essential quality criteria and be considered complete."
}
```